# Optimización de portafolio PV + BESS

### Qué decide este notebook

El modelo base (`01_Modelo base PV_BESS.ipynb`) responde a la pregunta *"dado un
sistema de 100 MW de PV y 30 MW / 120 MWh de BESS, ¿cómo conviene operarlo?"*.

Este notebook responde a una pregunta anterior y más importante:

> **¿Cuánto PV y cuánto BESS conviene instalar, y bajo qué regla de operación,
> para maximizar el valor económico del proyecto?**

Es decir: el tamaño del PV y del BESS dejan de ser un dato de entrada y pasan a
ser **variables de decisión**. La estrategia de operación también. El criterio de
selección es el **Valor Presente Neto (VPN)** del proyecto a 20 años.

---

### Por qué hizo falta un notebook nuevo

El diagnóstico del modelo base encontró que sus tres heurísticas producían
resultados prácticamente idénticos (0,15 % de diferencia). Las causas fueron:

| Causa | Evidencia |
|---|---|
| La tarifa regulada es plana | Spread cara/barata de **1,03 %**, cuando la eficiencia de ciclo exige **11,1 %** para que el arbitraje pague |
| El BESS está saturado | El excedente PV supera la capacidad útil **88 %** de los días; el déficit la supera el **100 %** |
| La degradación costaba cero | Ciclar era gratis, así que ninguna regla se penalizaba por ciclar de más |
| El BESS no podía exportar | Su única salida era la demanda, cuyo valor es casi constante hora a hora |

Este notebook corrige las cuatro cosas. Además incorpora un hallazgo posterior
que conviene tener presente desde el principio:

> **Habilitar la exportación desde el BESS no cambió nada por sí sola.** Alimentar
> la demanda ahorra ~656 COP/kWh mientras que exportar solo ingresa ~166 COP/kWh, y
> siempre hay más déficit del que la batería puede cubrir. La batería nunca
> encuentra rentable exportar *en la configuración base*.
>
> Lo que sí cambia el panorama es el **dimensionamiento**: con más PV y más
> batería, el excedente deja de caber en la demanda y la exportación pasa a ser
> la salida dominante. Estrategia y tamaño están acoplados y no se pueden
> escoger por separado. Ese acoplamiento es el objeto de este notebook.

---

### Estructura

1. Librerías
2. Datos de entrada y normalización del perfil solar
3. Señales de precio: tarifa regulada vs. indexada a bolsa
4. La degradación como **precio sombra** (y por qué no es un costo de caja)
5. Motor de despacho y las cuatro estrategias
6. Modelo financiero: VPN, TIR, payback y costo nivelado
7. Espacio de configuraciones
8. Barrido completo
9. Resultados: mejor configuración, mapas de VPN, valor marginal
10. Análisis de sensibilidad
11. Conclusiones

> **Tiempo de ejecución:** unos 3–5 minutos, dominados por el barrido de la
> sección 8 (unas 700 simulaciones anuales de 8.760 horas).

> Corrección tarifaria: punta N3 06:00–21:00 = 659,9496 COP/kWh; fuera de punta = 653,2247. T1 se recalibra. Regeneración integral completada con T0 corregida y T1 recalibrada.


## 1. Librerías

Solo se necesitan numpy, pandas y matplotlib. A diferencia del modelo base, aquí
**no se usa PyPSA**: el barrido evalúa cientos de configuraciones y un MILP anual
por configuración sería inviable. En su lugar se usan reglas heurísticas rápidas
(~0,2 s por año simulado), que es exactamente el caso de uso para el que sirven.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

# ------------------------------------------------------------
# Paleta categórica (validada para daltonismo, orden fijo).
# El orden NO se cicla: cada estrategia conserva su color en
# todas las gráficas del notebook.
# ------------------------------------------------------------
COLOR_ESTRATEGIA = {
    'A Autoconsumo':    '#2a78d6',   # azul
    'B Merito diario':  '#eb6834',   # naranja
    'C Merchant':       '#1baf7a',   # aqua
    'D Merito+Export':  '#eda100',   # amarillo
}
TINTA        = '#0b0b0b'   # texto principal
TINTA_SUAVE  = '#52514e'   # texto secundario
GRIS_REJILLA = '#d8d8d4'

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.grid': True,
    'grid.color': GRIS_REJILLA,
    'grid.linewidth': 0.6,
    'axes.edgecolor': GRIS_REJILLA,
    'axes.labelcolor': TINTA_SUAVE,
    'xtick.color': TINTA_SUAVE,
    'ytick.color': TINTA_SUAVE,
    'text.color': TINTA,
    'axes.titlecolor': TINTA,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
})

print('Librerías cargadas.')

Librerías cargadas.


## 2. Datos de entrada

Se reutilizan exactamente los mismos archivos del modelo base, de modo que los
resultados sean comparables:

| Archivo | Contenido |
|---|---|
| `datos/Oriental_tmy-2025.csv` | Año meteorológico típico de NSRDB para la región Oriente. La generación se simula con `solar_tmy.py`. **Es la fuente solar del modelo.** |
| `datos/Oriente.xlsx` | Serie histórica de potencia. Ya no alimenta el modelo; se conserva como contraste en §6.5 |
| `datos/Precio_Bolsa_Nacional_($kwh)_2025.xlsx` | Precio horario de bolsa 2025 |
| `datos/load variable.csv` | Perfil horario de demanda (separador `;`). Se limpia con `demanda.limpiar()`: 79 horas imputadas de 11 cortes de medidor. **Es la demanda del modelo, la misma que usa 01B.** |

### 2.1 Por qué hay que normalizar el perfil solar

La generación se simula para una planta de tamaño fijo (nominal 100 MWp).
Como aquí el tamaño del PV es una **variable de decisión**, el perfil se
convierte a **por unidad**: se divide entre el nominal de referencia
para obtener una serie de factor de planta horario.

$$\text{PV}_h(S) = \text{perfil}_{pu,h} \times S$$

donde $S$ es la capacidad instalada en MW. Esto supone que el perfil de
generación por MW instalado no cambia con el tamaño de la planta — válido
mientras la tecnología, la orientación y el sitio sean los mismos.

In [2]:
import solar_tmy as ST
import demanda as DEM

# El recurso solar se SIMULA desde el TMY de NSRDB, no se lee de un Excel de
# potencia. Es la misma fuente que usa 01B y el resto del proyecto; antes este
# cuaderno leia `Oriente.xlsx`, lo que lo dejaba con un recurso distinto al de
# su propia caracterizacion. El impacto del cambio se mide en la seccion 6.5.
REGION_SOLAR = 'Oriente'
PV_NOMINAL_REFERENCIA_MW = 100.0   # MWp DC, la unidad del CAPEX solar
ANIO = 2025

# Validacion historica de robustez frente a los datos de entrada (seccion 10.4).
# Recalcula dos busquedas completas sobre combinaciones de entradas ya retiradas,
# asi que anade unos 10 minutos. El barrido principal no depende de ella.
RUN_VALIDACION_HISTORICA = False


def buscar(*nombres):
    # Las entradas viven en datos/; la raíz queda como respaldo.
    for carpeta in (Path.cwd() / 'datos', Path.cwd()):
        for nombre in nombres:
            p = carpeta / nombre
            if p.exists():
                return p
    raise FileNotFoundError(f'No se encontró: {nombres}')


def leer_precio_bolsa(archivo, anio):
    """El archivo trae dos filas de encabezado y las 24 horas en columnas."""
    df = pd.read_excel(archivo, header=2)
    df['Fecha'] = pd.to_datetime(df['Fecha'], errors='coerce')
    df = df.dropna(subset=['Fecha'])

    horas = [str(h) for h in range(24)]
    df = df.melt(id_vars=['Fecha'], value_vars=horas,
                 var_name='Hora', value_name='Bolsa_COP_kWh')
    df['Hora'] = df['Hora'].astype(int)
    df['Fecha'] = df['Fecha'] + pd.to_timedelta(df['Hora'], unit='h')
    df['Bolsa_COP_kWh'] = pd.to_numeric(df['Bolsa_COP_kWh'], errors='raise')

    return (df.loc[df['Fecha'].dt.year == anio, ['Fecha', 'Bolsa_COP_kWh']]
              .sort_values('Fecha').reset_index(drop=True))


def perfil_pv_pu_tmy(region, nominal_MWp):
    """Perfil solar en por unidad, simulado desde el TMY de NSRDB con pvlib.

    Es la MISMA fuente que usa `01B_Caracterizacion_dimensionamiento.ipynb` y el
    resto del proyecto: `solar_tmy.perfil_pv()` recorre TMY -> posicion solar ->
    POA (Perez) -> IAM -> temperatura de celda -> PVWatts DC -> perdidas ->
    inversor -> AC. `nominal_MWp` es potencia DC, la misma unidad en que
    `costos_curvas.capex_pv_COP()` cobra el CAPEX solar.
    """
    d = ST.perfil_pv(region, nominal_MWp)
    return pd.DataFrame({'Fecha': d['Fecha'], 'PV_pu': d['PV_MW'] / nominal_MWp})


def leer_perfil_pv_pu_legado(archivo, anio, nominal_MW):
    """Serie historica de `Oriente.xlsx`. Ya NO alimenta el modelo.

    Se conserva unicamente para la prueba de impacto de la seccion 6.5, que
    documenta cuanto mueve el resultado el cambio de recurso.
    """
    df = pd.read_excel(archivo, sheet_name='Sheet1',
                       usecols=['date', 'Generacion_MW'])
    df['Fecha'] = pd.to_datetime(df['date'], errors='coerce')
    df['PV_pu_legado'] = (pd.to_numeric(df['Generacion_MW'], errors='coerce')
                            .clip(lower=0) / nominal_MW)

    return (df.loc[df['Fecha'].dt.year == anio, ['Fecha', 'PV_pu_legado']]
              .dropna().sort_values('Fecha').reset_index(drop=True))


def leer_demanda(archivo, anio, columna):
    df = pd.read_csv(archivo, sep=';')
    df['Fecha'] = pd.to_datetime(df['period'], dayfirst=True, errors='coerce')
    df['Load_MW'] = pd.to_numeric(df[columna], errors='coerce')

    return (df.loc[df['Fecha'].dt.year == anio, ['Fecha', 'Load_MW']]
              .dropna().sort_values('Fecha').reset_index(drop=True))


df_bolsa = leer_precio_bolsa(buscar('Precio_Bolsa_Nacional_($kwh)_2025.xlsx'), ANIO)
df_pv    = perfil_pv_pu_tmy(REGION_SOLAR, PV_NOMINAL_REFERENCIA_MW)
df_pv_leg = leer_perfil_pv_pu_legado(buscar('Oriente.xlsx'), ANIO,
                                     PV_NOMINAL_REFERENCIA_MW)
# La demanda pasa por `demanda.limpiar()`, igual que en 01B y en el cuaderno
# 01. Imputa los 71 cortes de medidor y sus horas de recuperacion con la mediana
# de la misma hora del dia en los dias validos a +/- 7 dias. La logica NO se
# duplica aqui: vive en `demanda.py` y es la unica implementacion del proyecto.
df_load_crudo = leer_demanda(buscar('load variable.csv'), ANIO, 'Porvenir')
df_load = DEM.limpiar(df_load_crudo)
LIMPIEZA = df_load.attrs['limpieza_demanda']

# La serie cruda se conserva SOLO para la prueba de impacto de la seccion 6.6.
df_load_crudo = df_load_crudo.rename(columns={'Load_MW': 'Load_MW_crudo'})

datos = (df_pv.merge(df_load, on='Fecha', how='inner', validate='one_to_one')
               .merge(df_bolsa, on='Fecha', how='inner', validate='one_to_one')
               .merge(df_pv_leg, on='Fecha', how='inner', validate='one_to_one')
               .merge(df_load_crudo, on='Fecha', how='inner', validate='one_to_one'))

assert len(datos) == 8760, f'Se esperaban 8.760 horas y hay {len(datos)}'

print(f'Horas: {len(datos):,}')
print(f'Recurso solar          : TMY NSRDB, region {REGION_SOLAR}, via solar_tmy.py')
print(f'Factor de planta solar : {datos.PV_pu.mean():.3f}  '
      f'({datos.PV_pu.sum():,.0f} MWh por MWp instalado al año)')
print(f'  serie de contraste   : Oriente.xlsx, FC {datos.PV_pu_legado.mean():.3f} '
      f'({datos.PV_pu_legado.sum():,.0f} MWh/MWp) — solo para la seccion 6.5')
print(f'Demanda                : media {datos.Load_MW.mean():.2f} MW | '
      f'máx {datos.Load_MW.max():.2f} MW | {datos.Load_MW.sum():,.0f} MWh/año')
print(f'  limpieza aplicada    : {LIMPIEZA["horas_imputadas"]} horas imputadas '
      f'({LIMPIEZA["horas_en_cero"]} cortes + {LIMPIEZA["horas_de_recuperacion"]} '
      f'de recuperación) en {LIMPIEZA["rachas"]} rachas')
print(f'  energía              : {LIMPIEZA["energia_antes_MWh"]:,.0f} -> '
      f'{LIMPIEZA["energia_despues_MWh"]:,.0f} MWh/año '
      f'({LIMPIEZA["incremento_%"]:+.2f} %)')
print(f'  NO se tocan          : {LIMPIEZA["horas_bajas_no_tocadas"]} horas bajas '
      f'dispersas (podrían ser paradas reales de planta)')
print(f'Precio de bolsa        : media {datos.Bolsa_COP_kWh.mean():.1f} COP/kWh | '
      f'p5 {datos.Bolsa_COP_kWh.quantile(.05):.1f} | p95 {datos.Bolsa_COP_kWh.quantile(.95):.1f}')

Horas: 8,760
Recurso solar          : TMY NSRDB, region Oriente, via solar_tmy.py
Factor de planta solar : 0.158  (1,387 MWh por MWp instalado al año)
  serie de contraste   : Oriente.xlsx, FC 0.156 (1,364 MWh/MWp) — solo para la seccion 6.5
Demanda                : media 18.12 MW | máx 23.23 MW | 158,743 MWh/año
  limpieza aplicada    : 79 horas imputadas (71 cortes + 8 de recuperación) en 11 rachas
  energía              : 157,306 -> 158,743 MWh/año (+0.91 %)
  NO se tocan          : 115 horas bajas dispersas (podrían ser paradas reales de planta)
Precio de bolsa        : media 240.8 COP/kWh | p5 104.8 | p95 704.2


### 2.1 La demanda es la misma que la de 01B

Los dos cuadernos tienen que compartir exactamente la misma serie horaria.
No basta con que ambos llamen a `demanda.limpiar()`: hay que comprobarlo
contra la serie que 01B usa de verdad, que llega por
`sensibilidad.cargar_entorno_notebook()` desde el cuaderno 01.

La comprobación verifica **identidad temporal y numérica**: mismas 8.760
marcas de tiempo, en el mismo orden, con los mismos valores.

In [3]:
# Se compara contra la serie que usa 01B de verdad, no contra una
# reconstruccion: `cargar_entorno_notebook()` ejecuta las celdas del
# cuaderno 01 y devuelve el caso ya armado. Tarda unos segundos.
import logging
for _r in ('pypsa', 'linopy'):
    logging.getLogger(_r).setLevel(logging.ERROR)
import sensibilidad as S

_g01 = S.cargar_entorno_notebook()
_load_01B = _g01['df_base'][['Fecha', 'Load_MW']].sort_values('Fecha')
_load_02 = datos[['Fecha', 'Load_MW']].sort_values('Fecha')

print('=' * 78)
print('IDENTIDAD DE LA DEMANDA CON 01B')
print('=' * 78)

assert len(_load_01B) == len(_load_02) == 8760, 'No son 8.760 horas en ambos'
assert (_load_01B.Fecha.to_numpy() == _load_02.Fecha.to_numpy()).all(), \
    'Las marcas de tiempo no coinciden'
assert np.allclose(_load_01B.Load_MW.to_numpy(),
                   _load_02.Load_MW.to_numpy(), rtol=0, atol=1e-9), \
    'La demanda de 02 no es identica a la de 01B'

_dif = np.abs(_load_01B.Load_MW.to_numpy() - _load_02.Load_MW.to_numpy()).max()
print(f'  Horas                 : {len(_load_02):,} en ambos')
print(f'  Marcas de tiempo      : identicas')
print(f'  Diferencia maxima     : {_dif:.2e} MW')
print(f'  Energia anual 01B     : {_load_01B.Load_MW.sum():,.2f} MWh')
print(f'  Energia anual 02      : {_load_02.Load_MW.sum():,.2f} MWh')
print()
print('Los dos cuadernos comparten ahora el sol y la demanda.')

del _g01   # el entorno del 01 no se usa para nada mas aqui

IDENTIDAD DE LA DEMANDA CON 01B
  Horas                 : 8,760 en ambos
  Marcas de tiempo      : identicas
  Diferencia maxima     : 0.00e+00 MW
  Energia anual 01B     : 158,742.57 MWh
  Energia anual 02      : 158,742.57 MWh

Los dos cuadernos comparten ahora el sol y la demanda.


## 3. Señales de precio

El proyecto compra energía a la red cuando no le alcanza y vende excedentes. Hacen
falta entonces dos series horarias:

- $c^{imp}_h$ — **costo de importar** 1 MWh en la hora $h$
- $c^{exp}_h$ — **ingreso neto por exportar** 1 MWh en la hora $h$

### 3.1 Ingreso por exportación

El excedente se liquida al precio de bolsa menos los cargos variables que aplican
a la energía exportada:

$$c^{exp}_h = P^{bolsa}_h - (\text{CERE} + \text{FAZNI} + \text{arranque/parada})$$

### 3.2 Costo de importación: dos estructuras tarifarias

Se comparan dos, porque **la estructura tarifaria cambia cuál es el tamaño óptimo**:

**T0 — Tarifa regulada (opción horaria CREG).** Dos bandas de precio casi iguales.
Es lo que aplica hoy a un usuario no regulado con esta opción.

**T1 — Tarifa indexada a bolsa.** El usuario paga el precio de bolsa más un margen
fijo que representa cargos de red, comercialización, pérdidas y restricciones:

$$c^{imp}_h = P^{bolsa}_h + m$$

El margen $m$ se calibra a **ingreso neutro**: se elige de modo que un usuario
*sin* proyecto pague exactamente lo mismo bajo T0 y bajo T1.

$$m = \frac{\sum_h c^{imp,T0}_h \cdot D_h - \sum_h P^{bolsa}_h \cdot D_h}{\sum_h D_h}$$

Esto es metodológicamente importante: sin la calibración estaríamos comparando una
tarifa *más barata* contra una *más cara*, y no sabríamos si las diferencias vienen
de la **volatilidad** o del **nivel**. Con la calibración, cualquier diferencia es
atribuible únicamente a la forma horaria de la señal.

In [4]:
# ------------------------------------------------------------
# Parámetros tarifarios (Res. CREG 119/2007 y 079/1997, nivel 3)
# ------------------------------------------------------------
TARIFA_CARA_COP_kWh   = 659.9496
TARIFA_BARATA_COP_kWh = 653.2247
HORAS_CARAS_CREG      = set(range(6, 21))  # Punta N3: 06:00 <= hora < 21:00

# Cargos variables sobre la energía exportada
CERE_COP_MWh              = 67_811.57
FAZNI_COP_kWh             = 3.2695
ARRANQUE_PARADA_COP_kWh   = 4.0

datos['Hora'] = datos.Fecha.dt.hour
bolsa_COP_MWh = datos.Bolsa_COP_kWh.to_numpy() * 1000

# --- Ingreso por exportación ---
cargos_export_COP_MWh = (CERE_COP_MWh
                         + FAZNI_COP_kWh * 1000
                         + ARRANQUE_PARADA_COP_kWh * 1000)
c_exp = bolsa_COP_MWh - cargos_export_COP_MWh

# --- T0: punta N3 diurna; valores confirmados por usuario, pliego/mes pendiente ---
c_imp_T0 = np.where(datos.Hora.isin(HORAS_CARAS_CREG),
                    TARIFA_CARA_COP_kWh * 1000,
                    TARIFA_BARATA_COP_kWh * 1000)

# --- T1: indexada, calibrada a ingreso neutro sobre la demanda ---
demanda = datos.Load_MW.to_numpy()
margen_COP_MWh = ((c_imp_T0 * demanda).sum() - (bolsa_COP_MWh * demanda).sum()) / demanda.sum()
c_imp_T1 = bolsa_COP_MWh + margen_COP_MWh

TARIFAS = {'T0 — señal cuasi-plana': c_imp_T0, 'T1 — señal horaria indexada': c_imp_T1}

print(f'Margen de la tarifa indexada: {margen_COP_MWh/1000:,.2f} COP/kWh')
print()
resumen_tarifas = pd.DataFrame({
    n: {'mín [COP/kWh]': v.min()/1000, 'media [COP/kWh]': v.mean()/1000,
        'máx [COP/kWh]': v.max()/1000, 'rango máx/mín': v.max()/v.min(),
        'costo demanda [mil MM COP]': (v*demanda).sum()/1e9}
    for n, v in TARIFAS.items()}).T
display(resumen_tarifas.style.format({
    'mín [COP/kWh]': '{:,.1f}', 'media [COP/kWh]': '{:,.1f}',
    'máx [COP/kWh]': '{:,.1f}', 'rango máx/mín': '{:,.2f}x',
    'costo demanda [mil MM COP]': '{:,.2f}'}))
print('\nEl costo para un usuario sin proyecto es idéntico: la calibración funcionó.')

Margen de la tarifa indexada: 412.39 COP/kWh



,mín [COP/kWh],media [COP/kWh],máx [COP/kWh],rango máx/mín,costo demanda [mil MM COP]
T0 — señal cuasi-plana,653.2,657.4,659.9,1.01x,104.36
T1 — señal horaria indexada,510.2,653.2,"2,636.4",5.17x,104.36



El costo para un usuario sin proyecto es idéntico: la calibración funcionó.


In [5]:
# Comparación visual de las dos señales sobre una semana representativa
ventana = slice(24*65, 24*72)
fig, ax = plt.subplots(figsize=(13, 4))

ax.plot(datos.Fecha[ventana], c_imp_T0[ventana]/1000,
        color='#2a78d6', linewidth=2, label='T0 — señal cuasi-plana')
ax.plot(datos.Fecha[ventana], c_imp_T1[ventana]/1000,
        color='#eb6834', linewidth=2, label='T1 — señal horaria indexada')

ax.set_title('Costo de importación: una señal plana y una señal viva',
             loc='left', fontweight='bold')
ax.set_ylabel('COP/kWh')
ax.legend(frameon=False, loc='upper left')
ax.grid(axis='x', visible=False)
plt.tight_layout()
plt.show()

print(f'Rango T0: {c_imp_T0.max()/c_imp_T0.min():.3f}x    '
      f'Rango T1: {c_imp_T1.max()/c_imp_T1.min():.1f}x')
print('Con T0 la hora del día es económicamente irrelevante. Con T1, no.')

Rango T0: 1.010x    Rango T1: 5.2x
Con T0 la hora del día es económicamente irrelevante. Con T1, no.


## 4. La degradación como precio sombra

Este es el punto conceptual más delicado del notebook, así que conviene detenerse.

Cada MWh que entra y sale de la batería consume vida útil. Hay dos formas de
representarlo y **mezclarlas es un error de doble conteo**:

| Enfoque | Cómo entra la degradación | Uso |
|---|---|---|
| **Contable** | El CAPEX ya paga la batería; la degradación aparece como **reposición** en el año en que se agotan los ciclos | Flujo de caja / VPN |
| **Operativo** | Un costo por MWh de throughput que representa la vida consumida | Decisión de despacho |

Si se cobra el CAPEX *y además* un costo de degradación en el flujo de caja, se
paga la batería dos veces.

La solución que usa este notebook:

> El costo de degradación es un **precio sombra que solo guía el despacho**. Entra
> en las reglas de operación para evitar ciclos que no se pagan solos, pero **no
> entra en el flujo de caja**. En el flujo de caja la batería aparece como CAPEX y,
> si sus ciclos se agotan antes del horizonte, como una reposición.

El valor del precio sombra es el costo de capital amortizado por unidad de energía
que atraviesa la batería:

$$c_{deg} = \frac{\text{CAPEX}_{BESS}}{N_{ciclos} \times (E_{util} + E_{util}/\eta_{RT})}$$

El denominador cuenta la energía que *sale* más la que *entra* (que es mayor por
las pérdidas), porque el throughput se mide en ambos sentidos.

In [6]:
# ------------------------------------------------------------
# Parámetros técnicos del BESS
# ------------------------------------------------------------
DoD    = 0.80    # profundidad de descarga
ETA_RT = 0.90    # eficiencia de ida y vuelta (round-trip)
DT     = 1.0     # paso horario [h]

ETA_C = ETA_D = np.sqrt(ETA_RT)   # se reparte simétricamente

# ------------------------------------------------------------
# Costos de inversión: se heredan de `costos.py`, que es la referencia
# citada del proyecto. Este cuaderno llevaba una copia con TRM 4.200, vida
# de 12 años y el OPEX del BESS en USD/kW-año en vez de USD/kWh-año, que
# equivalía a la mitad de la referencia sin que se notara en las tablas.
# ------------------------------------------------------------
import costos

TRM_COP_USD = costos.TRM_COP_USD

# El CAPEX ya no es plano: sale de las curvas de `CurvasCapexPVBESS.xlsx`,
# donde el costo unitario depende del tamaño y de la duración.
# Constantes planas historicas. Tras C1.3 ya NO alimentan ningun calculo del
# cuaderno: se conservan solo para poder contrastar el orden de magnitud contra
# las curvas del Excel.
CAPEX_PV_USD_kW = costos.CAPEX_PV_USD_KW
CAPEX_BESS_ENERGIA_USD_kWh = costos.CAPEX_BESS_ENERGIA_USD_KWH
CAPEX_BESS_POTENCIA_USD_kW = costos.CAPEX_BESS_POTENCIA_USD_KW

# C1.6: `OPEX_BESS_USD_kW_anio = OPEX_BESS_USD_KWH_ANIO * 4.0` se elimino. Era
# codigo muerto -ningun calculo lo leia- y ademas escondia otra suposicion de
# 4 h. El OPEX real sale de `costos.opex_bess_anual()`, que es el 1,5 % del CAPEX.

CICLOS_VIDA_BESS = costos.CICLOS_VIDA_BESS
VIDA_CALENDARIO_BESS = costos.VIDA_CALENDARIO_BESS


def costo_degradacion_COP_MWh(power_MW, cap_MWh):
    """Precio sombra del throughput. Solo guía el despacho, no el flujo de caja.

    C1.1: antes se llamaba `costos.capex_bess(cap_MWh / 4.0, cap_MWh)`, que
    imponía una geometría de 4 h al CAPEX que se amortiza. Con potencia y
    energía ya independientes eso desviaba el precio sombra hasta -28 % en las
    baterías cortas y +20 % en las largas. Ahora recibe la potencia real.

    El CAPEX que aparece aquí es el que se usa para repartir el desgaste; el
    CAPEX financiero del proyecto lo calcula `evaluar_finanzas` por separado y
    siempre usó la potencia real. Ciclos de vida, DoD, RTE y la definición de
    throughput no cambian.
    """
    if cap_MWh <= 0 or power_MW <= 0:
        return 0.0
    capex = costos.capex_bess(power_MW, cap_MWh)
    util = cap_MWh * DoD
    throughput_vida = CICLOS_VIDA_BESS * (util + util / ETA_RT)
    return capex / throughput_vida


def _c_deg_geometria_4h(cap_MWh):
    """Version ANTERIOR, solo para documentar el cambio. No se usa en el modelo."""
    if cap_MWh <= 0:
        return 0.0
    util = cap_MWh * DoD
    return costos.capex_bess(cap_MWh / 4.0, cap_MWh) / (
        CICLOS_VIDA_BESS * (util + util / ETA_RT))


print('=' * 88)
print('C1.1 — PRECIO SOMBRA: ANTES (geometría 4 h forzada) vs DESPUÉS (geometría real)')
print('=' * 88)
print(f"{'Configuración':<22}{'dur [h]':>9}{'antes':>14}{'después':>14}{'cambio':>10}")
for _p, _e in [(20.0, 80.0), (60.0, 80.0), (5.0, 40.0), (40.0, 320.0),
               (10.0, 40.0), (30.0, 120.0)]:
    _a = _c_deg_geometria_4h(_e)
    _d = costo_degradacion_COP_MWh(_p, _e)
    print(f'{f"{_p:.0f} MW / {_e:.0f} MWh":<22}{_e/_p:>9.2f}'
          f'{_a:>14,.0f}{_d:>14,.0f}{_d/_a - 1:>9.1%}')

print()
print('En COP/MWh de throughput. A 4 h exactas no cambia nada, que es la')
print('comprobación de que el único origen del cambio es el CAPEX que se amortiza.')
print()
print('El precio sombra NO es independiente del tamaño: el CAPEX viene de una')
print('curva por segmentos de energía, así que el costo unitario cambia con la')
print('escala y con la geometría.')

C1.1 — PRECIO SOMBRA: ANTES (geometría 4 h forzada) vs DESPUÉS (geometría real)
Configuración           dur [h]         antes       después    cambio


20 MW / 80 MWh             4.00        43,653        43,653     0.0%
60 MW / 80 MWh             1.33        43,653        60,305    38.1%
5 MW / 40 MWh              8.00        45,914        42,193    -8.1%
40 MW / 320 MWh            8.00        52,397        43,656   -16.7%
10 MW / 40 MWh             4.00        45,914        45,914     0.0%
30 MW / 120 MWh            4.00        54,008        54,008     0.0%

En COP/MWh de throughput. A 4 h exactas no cambia nada, que es la
comprobación de que el único origen del cambio es el CAPEX que se amortiza.

El precio sombra NO es independiente del tamaño: el CAPEX viene de una
curva por segmentos de energía, así que el costo unitario cambia con la
escala y con la geometría.


## 5. Motor de despacho

### 5.1 El balance horario: qué compite con qué

En cada hora el sistema resuelve un pequeño problema de asignación. Primero, la
generación solar cubre la demanda hasta donde alcanza — esto nunca es discutible,
porque evita una compra y no cuesta nada:

```
PV_a_demanda = min(PV, Demanda)
excedente    = PV - PV_a_demanda      (>= 0)
deficit      = Demanda - PV           (>= 0)
```

Nótese que **excedente y déficit nunca son positivos a la vez**. Eso parte la hora
en dos casos mutuamente excluyentes:

**Caso A — hora de excedente** (sobra sol). Destinos posibles del excedente:

| Destino | Valor |
|---|---|
| Cargar el BESS | Guardar para después, a cambio de perder $(1-\eta)$ |
| Exportar a la red | $c^{exp}_h$ ahora mismo |
| Verter | 0 (solo si exportar da pérdida) |

**Caso B — hora de déficit** (falta sol). Formas de cubrirlo:

| Origen | Costo |
|---|---|
| Descargar el BESS | El valor de oportunidad de esa energía guardada |
| Importar de la red | $c^{imp}_h$ |

La regla de operación es, precisamente, **cómo se resuelve esa asignación**.

### 5.2 Restricciones que respetan todas las estrategias

1. Límites de estado de carga: $SOC_{min} \le SOC_h \le SOC_{max}$, con $SOC_{min} = E(1-DoD)$
2. Límite de potencia: carga y descarga $\le P_{nom}$
3. No hay carga y descarga simultáneas
4. **No hay importación y exportación simultáneas** en el punto de conexión
5. **SOC cíclico**: el estado de carga al final del año debe igualar al del inicio,
   para que ninguna estrategia se beneficie de vaciar la batería al terminar

La condición 5 se impone por punto fijo: se simula el año, se toma el SOC final
como nuevo SOC inicial y se repite hasta que converge.

### Dos sentidos distintos de «óptimo»

El cuaderno usa la palabra en dos lugares y conviene no mezclarlos:

| Término | Qué significa aquí |
|---|---|
| **Mejor configuración encontrada en la malla heurística** | El máximo de VPN sobre las 233 configuraciones evaluadas. La búsqueda de *dimensionamiento* sí es exhaustiva, pero solo sobre esa malla discreta: fuera de ella no se sabe nada |
| **Despacho matemáticamente óptimo** | La operación que un solver demostraría insuperable. **Este cuaderno no la calcula** |

Las cuatro estrategias A–D son **reglas heurísticas**: deciden hora a hora con
criterios fijos y un cierre cíclico de SOC por iteración de punto fijo. No hay
PyPSA, ni linopy, ni ningún solver MILP en este cuaderno.

Consecuencia práctica: cuando aquí se lee «configuración óptima», hay que
entender **la mejor de las evaluadas, operada con una regla heurística**. La
cota de despacho óptimo con previsión perfecta vive en
`01_Modelo base PV_BESS.ipynb`, que sí resuelve un MILP con PyPSA.

### 5.3 Las cuatro estrategias

Las estrategias se diferencian en **una sola decisión**: a qué horas asignan la
energía almacenada. Todas cargan con excedente solar, porque esa energía no tiene
costo de oportunidad más allá de la exportación que se deja de hacer.

---

#### A — Autoconsumo *(greedy, sin previsión)*

> Carga siempre que sobre sol. Descarga siempre que falte.

Es la regla de referencia y la más común en la práctica industrial. No necesita
pronóstico de precio ni de demanda: solo mira la hora actual. En la literatura de
autoconsumo fotovoltaico es la regla estándar (Luthander et al., 2015).

**Debilidad esperada:** gasta la batería en la primera hora de déficit que
aparece, aunque más tarde ese mismo día haya horas mucho más caras.

---

#### B — Mérito diario *(umbral volumétrico, con previsión day-ahead)*

> Descarga solo en las horas de déficit **más caras del día**, tantas como
> alcance a cubrir la energía disponible.

Cada día se ordenan las horas de déficit de mayor a menor precio y se acumula la
capacidad de descarga hasta agotar la energía que se espera tener almacenada. El
precio de la última hora incluida es el **umbral**: descargar solo por encima de él
concentra la batería donde más ahorra.

Formalmente, el umbral $\tau_d$ del día $d$ satisface:

$$\sum_{h \in d\,:\,c^{imp}_h \ge \tau_d} \min(\text{déficit}_h, P_{nom}) \approx E_{disponible,d}$$

Requiere pronóstico de precio y demanda a 24 h, algo razonable en la práctica.
Es la versión heurística de un despacho con previsión y el puente natural hacia un
esquema de control predictivo (MPC).

---

#### C — Merchant *(arbitraje puro)*

> Compra barato, vende caro. Trata la batería como un activo de mercado y no como
> un complemento del autoconsumo.

Carga desde la red cuando el precio está bajo el cuartil inferior del día, y
descarga (a demanda o a red, lo que pague más) cuando supera el cuartil superior.
Solo actúa si el diferencial esperado cubre las pérdidas de ciclo y la degradación:

$$P^{max}_d \cdot \eta_{RT} - c^{imp}_h - c_{deg} > 0$$

Es el modelo de negocio de una batería independiente (Sioshansi et al., 2009).
Se incluye para responder una pregunta concreta: **¿conviene operar como merchant
cuando además hay una demanda que atender?**

---

#### D — Mérito + exportación

> Como B, pero además exporta desde el BESS cuando el precio de exportación supera
> el valor de guardar esa energía para la demanda.

$$c^{exp}_h - c_{deg} > \max(\tau_d,\, 0)$$

Es la estrategia que puede aprovechar las configuraciones con mucho PV, donde el
excedente desborda la demanda y la red pasa a ser la salida principal.

---

**Predicción antes de correr:** bajo T0 (tarifa plana) A ≈ B ≈ D, porque no hay
señal horaria que explotar. Bajo T1 y con PV grande, D debería separarse. C debería
perder en todos los casos donde haya demanda que atender, porque sacrifica un
ahorro de ~656 COP/kWh por un arbitraje de mucho menos. Veamos si se cumple.

In [7]:
ESTRATEGIAS = ['A Autoconsumo', 'B Merito diario', 'C Merchant', 'D Merito+Export']

# Índices de cada día del año: [(inicio, fin), ...]
DIA_IDX = [(i, i + 24) for i in range(0, 8760, 24)]
DIA_KEY = np.repeat(np.arange(len(DIA_IDX)), 24)


def umbral_volumetrico(deficit, c_imp, energia_dia, power_MW):
    """Umbral diario de precio que agota exactamente la energía disponible.

    Para cada día ordena las horas de déficit por precio descendente y acumula
    la energía descargable hasta cubrir `energia_dia`. Devuelve, para cada hora,
    el precio de corte de su día. Si la energía alcanza para todas las horas de
    déficit, el umbral es -inf (equivale a descargar siempre).
    """
    umbral = np.full(len(deficit), -np.inf)

    for k, (a, b) in enumerate(DIA_IDX):
        precio_dia = c_imp[a:b]
        capacidad_h = np.minimum(deficit[a:b], power_MW)

        orden = np.argsort(-precio_dia)
        acumulado = np.cumsum(capacidad_h[orden])
        corte = np.searchsorted(acumulado, energia_dia[k])

        if corte < len(orden):
            umbral[a:b] = precio_dia[orden[corte]]

    return umbral

In [8]:
# F no entra en `ESTRATEGIAS`: los barridos y el cuaderno 09 iteran esa lista
# y la recogerían sin pedirlo. Se declara aparte y se pasa por nombre.
ESTRATEGIA_F = 'F Merchant reserva'


def despachar(pv_MW, load_MW, c_imp, c_exp, estrategia,
              power_MW, cap_MWh, c_deg=0.0,
              p_exp_max=np.inf, p_imp_max=np.inf,
              max_iter=25, tol_ciclico=1e-4):
    """Simula un año de operación bajo una regla dada.

    Devuelve un DataFrame horario con el despacho completo. El SOC se cierra
    de forma cíclica por iteración de punto fijo.

    Simplificación deliberada: el SOC es un escalar único y no se rastrea el
    origen de la energía almacenada. C Merchant puede cargar desde el PV y
    desde la red el mismo día, así que la energía que exporta no es
    atribuible a una de las dos fuentes. Separarlas exigiría dos estados de
    carga; no se hace, porque el modelo compara reglas de operación y no
    liquida contratos.

    `F Merchant reserva` es C con una sola restricción añadida: la carga desde
    la red de cada día no puede ocupar el espacio que el excedente PV previsto
    de ese día va a necesitar. Todo lo demás —selección de descarga, condición
    de exportación, carga desde PV y condición económica de compra— es idéntico
    a C, para que la comparación aísle la reserva.
    """
    SOC_max = cap_MWh
    SOC_min = cap_MWh * (1 - DoD)
    usable = SOC_max - SOC_min

    pv_a_load = np.minimum(pv_MW, load_MW)
    excedente = pv_MW - pv_a_load
    deficit = load_MW - pv_a_load
    n = len(pv_MW)

    # --- Pronóstico day-ahead de energía almacenable ---
    exc_dia = np.array([excedente[a:b].sum() for a, b in DIA_IDX])
    energia_dia = np.minimum(usable, exc_dia * ETA_C)

    # Reserva solar de F. `energia_dia` ya está en unidades de SOC: es el
    # excedente del día llevado a MWh almacenados con ETA_C. Lo que sobra de la
    # energía útil es el único espacio que F puede comprarle a la red ese día.
    cupo_red_dia = np.maximum(0.0, usable - energia_dia)

    # --- Umbrales por estrategia ---
    if estrategia in ('B Merito diario', 'D Merito+Export'):
        umbral = umbral_volumetrico(deficit, c_imp, energia_dia, power_MW)
    else:
        umbral = np.full(n, -np.inf)

    # El valor de descargar una hora es el mayor entre evitar una compra y
    # vender: max(c_imp, c_exp). C Merchant carga en las horas baratas del día
    # y descarga —a demanda o exportando— en las de valor alto, medidas sobre
    # esa misma serie. Antes el umbral de descarga se tomaba sobre c_imp y la
    # exportación solo exigía cubrir c_deg, así que la venta quedaba
    # desacoplada de la compra que la había justificado.
    s_imp = pd.Series(c_imp)
    valor_desc = np.maximum(c_imp, c_exp)
    s_val = pd.Series(valor_desc)
    q_bajo = s_imp.groupby(DIA_KEY).transform(lambda x: x.quantile(0.25)).to_numpy()
    q_alto_valor = s_val.groupby(DIA_KEY).transform(lambda x: x.quantile(0.75)).to_numpy()
    valor_max_dia = s_val.groupby(DIA_KEY).transform('max').to_numpy()

    soc_inicial = SOC_min

    for _ in range(max_iter):
        soc = float(soc_inicial)
        ch_pv = np.zeros(n); ch_red = np.zeros(n)
        dis_dem = np.zeros(n); dis_red = np.zeros(n); soc_serie = np.zeros(n)
        dia_actual, red_soc_dia = -1, 0.0

        for i in range(n):
            if DIA_KEY[i] != dia_actual:
                dia_actual, red_soc_dia = DIA_KEY[i], 0.0
            exc_i, def_i = excedente[i], deficit[i]
            espacio = SOC_max - soc
            disponible = soc - SOC_min
            a = b = c = d = 0.0

            if def_i > 1e-9:
                # ---------- Caso B: hora de déficit ----------
                if estrategia == 'A Autoconsumo':
                    descargar = True
                elif estrategia in ('B Merito diario', 'D Merito+Export'):
                    descargar = c_imp[i] >= umbral[i]
                else:
                    descargar = c_imp[i] >= q_alto_valor[i]

                if descargar and disponible > 1e-9 and c_imp[i] > c_deg:
                    c = min(def_i, power_MW, disponible * ETA_D / DT)
                    soc -= c / ETA_D * DT
                elif (estrategia in ('C Merchant', ESTRATEGIA_F)
                      and espacio > 1e-9
                      and c_imp[i] <= q_bajo[i]
                      and valor_max_dia[i] * ETA_RT - c_imp[i] - c_deg > 0):
                    techo = espacio
                    if estrategia == ESTRATEGIA_F:
                        techo = min(techo, max(
                            0.0, cupo_red_dia[DIA_KEY[i]] - red_soc_dia))
                    b = min(power_MW, techo / (ETA_C * DT),
                            max(0.0, p_imp_max - def_i))
                    soc += b * ETA_C * DT
                    red_soc_dia += b * ETA_C * DT

            else:
                # ---------- Caso A: hora de excedente ----------
                if exc_i > 1e-9 and espacio > 1e-9:
                    a = min(exc_i, power_MW, espacio / (ETA_C * DT))
                    soc += a * ETA_C * DT

                # Exportar desde el BESS si supera el valor de guardarlo.
                # C vende solo en las horas de valor alto del día, la misma
                # serie con la que decide cargar. D conserva su umbral
                # volumétrico, que ya cumplía esa función.
                if estrategia in ('C Merchant', ESTRATEGIA_F):
                    exportar = (c_exp[i] >= q_alto_valor[i]
                                and c_exp[i] - c_deg > 0.0)
                elif estrategia == 'D Merito+Export':
                    exportar = c_exp[i] - c_deg > max(umbral[i], 0.0)
                else:
                    exportar = False

                if (exportar and a < 1e-9 and disponible > 1e-9
                        and c_exp[i] > 0):
                    margen_exp = max(0.0, p_exp_max - (exc_i - a))
                    d = min(power_MW, disponible * ETA_D / DT, margen_exp)
                    soc -= d / ETA_D * DT

                # Carga desde la red en horas sin sol ni demanda
                if (estrategia in ('C Merchant', ESTRATEGIA_F)
                        and exc_i <= 1e-9
                        and a < 1e-9 and d < 1e-9 and espacio > 1e-9
                        and c_imp[i] <= q_bajo[i]
                        and valor_max_dia[i] * ETA_RT - c_imp[i] - c_deg > 0):
                    techo = espacio
                    if estrategia == ESTRATEGIA_F:
                        techo = min(techo, max(
                            0.0, cupo_red_dia[DIA_KEY[i]] - red_soc_dia))
                    b = min(power_MW, techo / (ETA_C * DT), p_imp_max)
                    soc += b * ETA_C * DT
                    red_soc_dia += b * ETA_C * DT

            ch_pv[i] = a; ch_red[i] = b; dis_dem[i] = c; dis_red[i] = d
            soc_serie[i] = soc

        if abs(soc - soc_inicial) <= tol_ciclico:
            break
        soc_inicial = float(np.clip(soc, SOC_min, SOC_max))

    sobrante = excedente - ch_pv
    export_pv = np.where(c_exp > 0, np.minimum(sobrante, p_exp_max), 0.0)

    df = pd.DataFrame({
        'PV_MW': pv_MW, 'Load_MW': load_MW, 'PV_a_Load_MW': pv_a_load,
        'Carga_PV_MW': ch_pv, 'Carga_Red_MW': ch_red, 'Carga_MW': ch_pv + ch_red,
        'Descarga_Demanda_MW': dis_dem, 'Descarga_Red_MW': dis_red,
        'Descarga_MW': dis_dem + dis_red, 'SOC_MWh': soc_serie,
        'Red_a_Load_MW': deficit - dis_dem,
        'Import_MW': deficit - dis_dem + ch_red,
        'Export_MW': export_pv + dis_red,
        'Vertimiento_MW': sobrante - export_pv,
    })
    df.attrs['SOC_inicial_MWh'] = soc_inicial
    df.attrs['cierre_ciclico_MWh'] = abs(soc - soc_inicial)
    return df


print('Motor de despacho definido.')

Motor de despacho definido.


### 5.4 Validación física

Ninguna cifra económica sirve si el despacho viola la física del sistema. Esta
función replica las mismas comprobaciones del modelo base y se ejecuta sobre cada
resultado antes de usarlo.

In [9]:
def validar(df, power_MW, cap_MWh, tol=1e-5):
    """Devuelve la lista de violaciones encontradas. Lista vacía = todo correcto."""
    problemas = []
    SOC_max, SOC_min = cap_MWh, cap_MWh * (1 - DoD)

    if cap_MWh > 0:
        if (df.SOC_MWh < SOC_min - tol).any():
            problemas.append('SOC por debajo del mínimo')
        if (df.SOC_MWh > SOC_max + tol).any():
            problemas.append('SOC por encima del máximo')
        if (df.Carga_MW > power_MW + tol).any():
            problemas.append('potencia de carga excedida')
        if (df.Descarga_MW > power_MW + tol).any():
            problemas.append('potencia de descarga excedida')
        if ((df.Carga_MW > tol) & (df.Descarga_MW > tol)).any():
            problemas.append('carga y descarga simultáneas')
        if df.attrs['cierre_ciclico_MWh'] > 1e-3:
            problemas.append(f"SOC no cíclico ({df.attrs['cierre_ciclico_MWh']:.2e} MWh)")

    if ((df.Import_MW > tol) & (df.Export_MW > tol)).any():
        problemas.append('importación y exportación simultáneas')

    balance_pv = (df.PV_MW - df.PV_a_Load_MW - df.Carga_PV_MW
                  - (df.Export_MW - df.Descarga_Red_MW) - df.Vertimiento_MW)
    if balance_pv.abs().max() > 1e-6:
        problemas.append(f'balance PV no cierra ({balance_pv.abs().max():.2e})')

    balance_dem = (df.Load_MW - df.PV_a_Load_MW
                   - df.Descarga_Demanda_MW - df.Red_a_Load_MW)
    if balance_dem.abs().max() > 1e-6:
        problemas.append(f'balance demanda no cierra ({balance_dem.abs().max():.2e})')

    if (df.Vertimiento_MW < -tol).any():
        problemas.append('vertimiento negativo')

    return problemas

In [10]:
# Prueba del motor sobre la configuración del modelo base, bajo ambas tarifas
PV_PU = datos.PV_pu.to_numpy()            # TMY: la fuente del modelo
PV_PU_LEGADO = datos.PV_pu_legado.to_numpy()   # Oriente.xlsx: solo para 6.5
LOAD  = datos.Load_MW.to_numpy()               # limpia: la serie del modelo
LOAD_CRUDO = datos.Load_MW_crudo.to_numpy()    # cruda: solo para 6.6

pruebas = []
for nombre_tarifa, c_imp in TARIFAS.items():
    for est in ESTRATEGIAS:
        d = despachar(PV_PU * 100, LOAD, c_imp, c_exp, est,
                      power_MW=30.0, cap_MWh=120.0,
                      c_deg=costo_degradacion_COP_MWh(30.0, 120.0))
        problemas = validar(d, 30.0, 120.0)
        pruebas.append({
            'Tarifa': nombre_tarifa, 'Estrategia': est,
            'Descarga [MWh]': d.Descarga_MW.sum(),
            'Export BESS [MWh]': d.Descarga_Red_MW.sum(),
            'Carga red [MWh]': d.Carga_Red_MW.sum(),
            'Ciclos/día': d.Descarga_MW.sum() / (120*DoD) / 365,
            'Precio medio descarga': (d.Descarga_Demanda_MW * c_imp).sum()
                                     / max(d.Descarga_Demanda_MW.sum(), 1) / 1000,
            'Validación': 'OK' if not problemas else ' | '.join(problemas),
        })

display(pd.DataFrame(pruebas).style.format({
    'Descarga [MWh]': '{:,.0f}', 'Export BESS [MWh]': '{:,.0f}',
    'Carga red [MWh]': '{:,.0f}', 'Ciclos/día': '{:.2f}',
    'Precio medio descarga': '{:,.1f}'}).hide(axis='index'))

print('\nLectura: con 100 MW de PV la batería nunca exporta, ni siquiera cuando se le permite.')
print('Alimentar la demanda paga más que exportar y siempre hay déficit de sobra.')

Tarifa,Estrategia,Descarga [MWh],Export BESS [MWh],Carga red [MWh],Ciclos/día,Precio medio descarga,Validación
T0 — señal cuasi-plana,A Autoconsumo,"31,689",0,0,0.90,659.0,OK
T0 — señal cuasi-plana,B Merito diario,"31,672",0,0,0.90,659.2,OK
T0 — señal cuasi-plana,C Merchant,"30,484",300,"5,834",0.87,659.9,OK
T0 — señal cuasi-plana,D Merito+Export,"31,747",180,0,0.91,659.2,OK
T1 — señal horaria indexada,A Autoconsumo,"31,689",0,0,0.90,761.1,OK
T1 — señal horaria indexada,B Merito diario,"31,497",0,0,0.90,769.6,OK
T1 — señal horaria indexada,C Merchant,"32,004",0,"21,083",0.91,770.4,OK
T1 — señal horaria indexada,D Merito+Export,"31,497",0,0,0.90,769.6,OK



Lectura: con 100 MW de PV la batería nunca exporta, ni siquiera cuando se le permite.
Alimentar la demanda paga más que exportar y siempre hay déficit de sobra.


## 6. Modelo financiero

Con el despacho resuelto, cada configuración produce un **flujo de caja anual**.

### 6.1 De la operación al flujo de caja

El beneficio operativo de un año es lo que el proyecto le ahorra al usuario frente
a comprar toda su demanda a la red:

$$\text{Beneficio}_{op} = \underbrace{\sum_h D_h \cdot c^{imp}_h}_{\text{costo sin proyecto}} - \underbrace{\left(\sum_h I_h \cdot c^{imp}_h - \sum_h E_h \cdot c^{exp}_h\right)}_{\text{costo con proyecto}}$$

donde $I_h$ es la importación y $E_h$ la exportación de la hora $h$.

### 6.2 Inversión y operación

$$\text{CAPEX} = S_{PV} \cdot c_{PV} + P_{BESS} \cdot c_{pot} + E_{BESS} \cdot c_{ene}$$

$$\text{OPEX}_{anual} = S_{PV} \cdot o_{PV} + P_{BESS} \cdot o_{BESS}$$

### 6.3 Degradación y reposición

- **PV:** pierde 0,5 % de producción al año. Se aplica como factor sobre el
  beneficio operativo.
- **BESS:** se agota por lo que ocurra primero, ciclos o calendario:

$$N_{años} = \min\left(\frac{N_{ciclos}}{\text{ciclos por año}},\; \text{vida calendario}\right)$$

  Al cumplirse, se paga la reposición del componente de energía.

### 6.4 Indicadores

$$\text{VPN} = -\text{CAPEX} + \sum_{t=1}^{T} \frac{F_t}{(1+r)^t} - \sum_{\text{reposiciones}} \frac{R}{(1+r)^{t_R}}$$

- **TIR:** la tasa $r$ que anula el VPN. Se resuelve por bisección.
- **Payback descontado:** primer año en que el acumulado descontado cruza cero.
- **Costo nivelado de suministro:** todo lo que cuesta atender la demanda,
  dividido por la demanda atendida, ambos descontados. Es el indicador que permite
  comparar contra la tarifa actual en COP/kWh.

> **Nota sobre la aproximación:** el beneficio operativo se calcula simulando el
> año 1 y escalándolo por la degradación del PV, en vez de re-simular los 20 años.
> Simular cada año de cada configuración multiplicaría el tiempo por 20. La
> aproximación es estándar y afecta por igual a todas las configuraciones, así que
> **no distorsiona la comparación**, que es lo que aquí interesa.

In [11]:
# ------------------------------------------------------------
# Parámetros financieros
# ------------------------------------------------------------
HORIZONTE_ANIOS   = 20
WACC              = 0.10     # tasa de descuento real en COP
DEGRADACION_PV    = 0.005    # 0,5 % anual


def tir(flujos, lo=-0.95, hi=3.0, tol=1e-8, max_iter=300):
    """Tasa interna de retorno por bisección (evita depender de numpy-financial)."""
    def vpn_a(r):
        return sum(f / (1 + r) ** t for t, f in enumerate(flujos))

    f_lo, f_hi = vpn_a(lo), vpn_a(hi)
    if np.isnan(f_lo) or np.isnan(f_hi) or f_lo * f_hi > 0:
        return np.nan

    for _ in range(max_iter):
        medio = (lo + hi) / 2
        f_medio = vpn_a(medio)
        if abs(f_medio) < tol:
            return medio
        if f_lo * f_medio < 0:
            hi, f_hi = medio, f_medio
        else:
            lo, f_lo = medio, f_medio
    return (lo + hi) / 2


def evaluar_finanzas(df, c_imp, c_exp, pv_MW, power_MW, cap_MWh):
    """Convierte un año de despacho en indicadores financieros a 20 años."""
    costo_sin_proyecto = (df.Load_MW.to_numpy() * c_imp).sum()
    costo_con_proyecto = ((df.Import_MW.to_numpy() * c_imp).sum()
                          - (df.Export_MW.to_numpy() * c_exp).sum())
    beneficio_anio1 = costo_sin_proyecto - costo_con_proyecto

    # --- Inversión ---
    # CAPEX y OPEX desde las curvas del Excel, via costos.py. El OPEX del
    # BESS es el 1,5 % de su CAPEX y el del PV el 2 %, como en el resto del
    # proyecto; antes eran valores absolutos por kW que no coincidian.
    capex = costos.capex_pv(pv_MW) + costos.capex_bess(power_MW, cap_MWh)
    opex = (costos.opex_pv_anual(pv_MW)
            + costos.opex_bess_anual(cap_MWh, power_MW=power_MW))

    # --- Vida del BESS y reposición ---
    if cap_MWh > 0:
        ciclos_anio = df.Descarga_MW.sum() / (cap_MWh * DoD)
        vida_bess = min(CICLOS_VIDA_BESS / max(ciclos_anio, 1e-9), VIDA_CALENDARIO_BESS)
        costo_reposicion = costos.capex_bess(power_MW, cap_MWh)
    else:
        ciclos_anio, vida_bess, costo_reposicion = 0.0, np.inf, 0.0

    # --- Flujos ---
    flujos = [-capex]
    for t in range(1, HORIZONTE_ANIOS + 1):
        flujo = beneficio_anio1 * (1 - DEGRADACION_PV) ** (t - 1) - opex
        if cap_MWh > 0 and vida_bess < HORIZONTE_ANIOS and t == int(np.ceil(vida_bess)):
            flujo -= costo_reposicion
        flujos.append(flujo)

    descuento = np.array([(1 + WACC) ** -t for t in range(len(flujos))])
    vpn = float(np.dot(flujos, descuento))

    acumulado = np.cumsum(np.array(flujos) * descuento)
    cruces = np.where(acumulado >= 0)[0]
    payback = int(cruces[0]) if len(cruces) else np.nan

    # --- Costo nivelado de suministro: NO REPORTAR ---
    # C1.4. La formula anterior multiplicaba (opex + costo_con_proyecto) por
    # (1 - DEGRADACION_PV)**t. Ni el OPEX ni las compras de red dependen del
    # envejecimiento de los paneles: el OPEX no baja porque el modulo se degrade,
    # y las compras de red deberian SUBIR, no bajar, al perder generacion solar.
    #
    # Corregirlo con rigor exige recalcular el despacho de cada ano con el PV ya
    # degradado, es decir un modelo multianual que no existe todavia (ver §10.8).
    # Antes que sostener una formula incorrecta o inventar una aproximacion, el
    # indicador se marca como no reportable.
    costo_nivelado = np.nan   # NO REPORTAR — pendiente de modelo multianual

    return {
        'Beneficio_anio1_COP': beneficio_anio1,
        'CAPEX_COP': capex, 'OPEX_anual_COP': opex,
        'VPN_COP': vpn, 'TIR': tir(flujos),
        'Payback_anios': payback,
        # Renombrado: es el costo nivelado de TODO el suministro del usuario,
        # no un LCOE de generacion.
        'Costo_nivelado_suministro_COP_kWh': costo_nivelado,
        'Ciclos_anio': ciclos_anio,
        'Vida_BESS_anios': min(vida_bess, HORIZONTE_ANIOS),
    }


print(f'Horizonte {HORIZONTE_ANIOS} años | WACC {WACC:.0%} | degradación PV {DEGRADACION_PV:.1%}/año')

Horizonte 20 años | WACC 10% | degradación PV 0.5%/año


## 6.5 Impacto del cambio de recurso solar

Este cuaderno leía `Oriente.xlsx`, una serie de potencia ya construida,
mientras que `01B` y el resto del proyecto simulan la generación desde la
irradiancia del TMY de NSRDB. Mismo sitio y misma nominal, pero **no la misma
serie**: la correlación horaria entre ambas es 0,83.

Antes de dar el cambio por hecho conviene medir cuánto mueve el resultado.
Se evalúa la mejor configuración encontrada hasta ahora —PV 70 MW +
BESS 20 MW / 80 MWh— con las dos series, bajo las cuatro estrategias y las
dos tarifas.

Un cambio pequeño en la energía anual puede producir uno grande en el
despacho: lo que decide la operación es **cuándo** llega la energía, no
cuánta llega al año.

In [12]:
PV_IMP, POT_IMP, CAP_IMP = 70.0, 20.0, 80.0


def caso_recurso(pv_pu, c_imp, estrategia):
    """Un ano completo con un perfil solar dado. No cambia ningun supuesto."""
    d = despachar(pv_pu * PV_IMP, LOAD, c_imp, c_exp, estrategia,
                  power_MW=POT_IMP, cap_MWh=CAP_IMP,
                  c_deg=costo_degradacion_COP_MWh(POT_IMP, CAP_IMP))
    fin = evaluar_finanzas(d, c_imp, c_exp, PV_IMP, POT_IMP, CAP_IMP)
    return {
        'Generacion PV [MWh]': d.PV_MW.sum(),
        'Autoconsumo directo [MWh]': d.PV_a_Load_MW.sum(),
        'Excedente PV [MWh]': (d.PV_MW - d.PV_a_Load_MW).sum(),
        'Carga BESS desde PV [MWh]': d.Carga_PV_MW.sum(),
        'Carga desde red [MWh]': d.Carga_Red_MW.sum(),
        'Descarga BESS [MWh]': d.Descarga_MW.sum(),
        'Importacion [MWh]': d.Import_MW.sum(),
        'Exportacion [MWh]': d.Export_MW.sum(),
        'Ciclos/ano': fin['Ciclos_anio'],
        'Beneficio ano 1 [mil MM]': fin['Beneficio_anio1_COP'] / 1e9,
        'VPN [mil MM]': fin['VPN_COP'] / 1e9,
    }


filas_imp = []
for nombre_tarifa, c_imp in TARIFAS.items():
    for est in ESTRATEGIAS:
        a = caso_recurso(PV_PU_LEGADO, c_imp, est)   # A: Oriente.xlsx
        b = caso_recurso(PV_PU, c_imp, est)          # B: TMY 2025
        for k in a:
            filas_imp.append({
                'Tarifa': nombre_tarifa, 'Estrategia': est, 'Magnitud': k,
                'A Oriente.xlsx': a[k], 'B TMY 2025': b[k],
                'Dif [%]': (b[k] / a[k] - 1) * 100 if a[k] != 0 else np.nan,
            })

IMPACTO = pd.DataFrame(filas_imp)

print('=' * 92)
print('IMPACTO DEL CAMBIO DE RECURSO SOLAR')
print(f'PV {PV_IMP:.0f} MW + BESS {POT_IMP:.0f} MW / {CAP_IMP:.0f} MWh, '
      f'{len(ESTRATEGIAS)} estrategias x {len(TARIFAS)} tarifas')
print('=' * 92)

orden = list(IMPACTO.Magnitud.unique())
resumen_imp = (IMPACTO.pivot_table(index='Magnitud', values='Dif [%]',
                                   aggfunc=['min', 'mean', 'max'])
                      .reindex(orden))
resumen_imp.columns = ['Dif min [%]', 'Dif media [%]', 'Dif max [%]']
display(resumen_imp.round(2))

print()
print('Detalle por tarifa y estrategia del VPN y del beneficio:')
clave = IMPACTO[IMPACTO.Magnitud.isin(['VPN [mil MM]', 'Beneficio ano 1 [mil MM]'])]
display(clave.pivot_table(index=['Tarifa', 'Estrategia'], columns='Magnitud',
                          values=['A Oriente.xlsx', 'B TMY 2025', 'Dif [%]'])
              .round(2))

print()
print('A partir de aqui el modelo usa SOLO el TMY. `Oriente.xlsx` queda como')
print('serie historica de contraste y no vuelve a entrar en ningun calculo.')

IMPACTO DEL CAMBIO DE RECURSO SOLAR
PV 70 MW + BESS 20 MW / 80 MWh, 4 estrategias x 2 tarifas


,Dif min [%],Dif media [%],Dif max [%]
Magnitud,,,
Generacion PV [MWh],1.68,1.68,1.68
Autoconsumo directo [MWh],-1.92,-1.92,-1.92
Excedente PV [MWh],7.21,7.21,7.21
Carga BESS desde PV [MWh],-1.08,0.49,1.28
Carga desde red [MWh],-0.75,-0.41,-0.08
Descarga BESS [MWh],-0.45,0.55,1.28
Importacion [MWh],1.12,1.21,1.37
Exportacion [MWh],9.74,14.96,16.41
Ciclos/ano,-0.45,0.55,1.28



Detalle por tarifa y estrategia del VPN y del beneficio:


A Oriente.xlsx                            B TMY 2025                               Dif [%]             
Magnitud                                    Beneficio ano 1 [mil MM] VPN [mil MM] Beneficio ano 1 [mil MM] VPN [mil MM] Beneficio ano 1 [mil MM] VPN [mil MM]
Tarifa                      Estrategia                                                                                                                       
T0 — señal cuasi-plana      A Autoconsumo                      53.90       218.06                    53.93       218.32                     0.06         0.12
                            B Merito diario                    53.89       218.01                    53.93       218.27                     0.06         0.12
                            C Merchant                         52.29       204.77                    52.26       204.57                    -0.05        -0.10
                            D Merito+Export                    53.90       218.02                    53.93       218.28                     0.06         0.12
T1 — señal horaria indexada A Autoconsumo                      54.24       220.86                    54.49       222.91                     0.46         0.93
                            B Merito diario                    55.13       228.22                    55.28       229.42                     0.26         0.53
                            C Merchant                         48.96       177.31                    49.03       177.95                     0.16         0.36
                            D Merito+Export                    55.13       228.22                    55.28       229.42                     0.26         0.53


A partir de aqui el modelo usa SOLO el TMY. `Oriente.xlsx` queda como
serie historica de contraste y no vuelve a entrar en ningun calculo.


## 6.6 Impacto de limpiar la demanda

La serie cruda tiene **71 horas exactamente en 0 MW** repartidas en 11 rachas
de 1 a 22 horas. No son consumo: son cortes de medidor, con demanda normal
antes y después. En varias rachas la recuperación es gradual, así que las
primeras horas tras el corte también están corruptas.

`demanda.limpiar()` imputa esas horas con la **mediana de la misma hora del
día** en los días válidos dentro de ±7 días. No promedia las horas anteriores
porque en la racha de 22 h eso dejaría un valor plano durante un ciclo diario
completo, borrando justo la variación día/noche que necesita un estudio de
despacho.

**Lo que no se toca:** las ~115 horas dispersas entre 0,7 y 8 MW alejadas de
los cortes. Podrían ser paradas reales de planta y rellenarlas sería fabricar
consumo. Quedan declaradas como incertidumbre conocida.

Se mide el efecto sobre la mejor configuración encontrada, igual que en §6.5.

In [13]:
def caso_demanda(load, c_imp, estrategia):
    """Un ano completo con una serie de demanda dada. Nada mas cambia."""
    d = despachar(PV_PU * PV_IMP, load, c_imp, c_exp, estrategia,
                  power_MW=POT_IMP, cap_MWh=CAP_IMP,
                  c_deg=costo_degradacion_COP_MWh(POT_IMP, CAP_IMP))
    fin = evaluar_finanzas(d, c_imp, c_exp, PV_IMP, POT_IMP, CAP_IMP)
    return {
        'Demanda anual [MWh]': d.Load_MW.sum(),
        'Demanda pico [MW]': d.Load_MW.max(),
        'PV a carga [MWh]': d.PV_a_Load_MW.sum(),
        'Excedente PV [MWh]': (d.PV_MW - d.PV_a_Load_MW).sum(),
        'Carga BESS desde PV [MWh]': d.Carga_PV_MW.sum(),
        'Carga desde red [MWh]': d.Carga_Red_MW.sum(),
        'Descarga BESS [MWh]': d.Descarga_MW.sum(),
        'Importacion [MWh]': d.Import_MW.sum(),
        'Exportacion [MWh]': d.Export_MW.sum(),
        'Ciclos/ano': fin['Ciclos_anio'],
        'Beneficio ano 1 [mil MM]': fin['Beneficio_anio1_COP'] / 1e9,
        'VPN [mil MM]': fin['VPN_COP'] / 1e9,
    }


filas_dem = []
for nombre_tarifa, c_imp in TARIFAS.items():
    for est in ESTRATEGIAS:
        a = caso_demanda(LOAD_CRUDO, c_imp, est)   # A: cruda
        b = caso_demanda(LOAD, c_imp, est)         # B: limpia
        for k in a:
            filas_dem.append({
                'Tarifa': nombre_tarifa, 'Estrategia': est, 'Magnitud': k,
                'A cruda': a[k], 'B limpia': b[k],
                'Dif abs': b[k] - a[k],
                'Dif [%]': (b[k] / a[k] - 1) * 100 if a[k] != 0 else np.nan,
            })

IMPACTO_DEM = pd.DataFrame(filas_dem)

print('=' * 92)
print('IMPACTO DE LIMPIAR LA DEMANDA')
print(f'PV {PV_IMP:.0f} MW + BESS {POT_IMP:.0f} MW / {CAP_IMP:.0f} MWh, '
      f'{len(ESTRATEGIAS)} estrategias x {len(TARIFAS)} tarifas')
print('=' * 92)

_orden = list(IMPACTO_DEM.Magnitud.unique())
_res = IMPACTO_DEM.groupby('Magnitud').agg(
    A_cruda=('A cruda', 'mean'), B_limpia=('B limpia', 'mean'),
    Dif_abs_media=('Dif abs', 'mean'),
    Dif_pct_min=('Dif [%]', 'min'), Dif_pct_media=('Dif [%]', 'mean'),
    Dif_pct_max=('Dif [%]', 'max')).reindex(_orden)
display(_res.round(2))

print()
print('Detalle del VPN y del beneficio por tarifa y estrategia:')
_clave = IMPACTO_DEM[IMPACTO_DEM.Magnitud.isin(
    ['VPN [mil MM]', 'Beneficio ano 1 [mil MM]'])]
display(_clave.pivot_table(index=['Tarifa', 'Estrategia'], columns='Magnitud',
                           values=['A cruda', 'B limpia', 'Dif [%]']).round(2))

print()
print('A partir de aqui el modelo usa SOLO la demanda limpia.')

IMPACTO DE LIMPIAR LA DEMANDA
PV 70 MW + BESS 20 MW / 80 MWh, 4 estrategias x 2 tarifas


,A_cruda,B_limpia,Dif_abs_media,Dif_pct_min,Dif_pct_media,Dif_pct_max
Magnitud,,,,,,
Demanda anual [MWh],157306.16,158742.56,1436.41,0.91,0.91,0.91
Demanda pico [MW],23.23,23.23,0.00,0.00,0.00,0.00
PV a carga [MWh],56088.68,56766.57,677.88,1.21,1.21,1.21
Excedente PV [MWh],41031.26,40353.37,-677.88,-1.65,-1.65,-1.65
Carga BESS desde PV [MWh],19772.35,19752.76,-19.59,-0.54,-0.12,0.25
Carga desde red [MWh],2550.55,2575.40,24.85,0.72,1.19,1.66
Descarga BESS [MWh],20090.61,20095.34,4.73,-0.24,0.02,0.58
Importacion [MWh],83709.92,84488.56,778.64,0.85,0.93,0.99
Exportacion [MWh],21291.41,20633.12,-658.29,-3.48,-3.19,-1.95



Detalle del VPN y del beneficio por tarifa y estrategia:


A cruda                              B limpia                               Dif [%]             
Magnitud                                    Beneficio ano 1 [mil MM] VPN [mil MM] Beneficio ano 1 [mil MM] VPN [mil MM] Beneficio ano 1 [mil MM] VPN [mil MM]
Tarifa                      Estrategia                                                                                                                       
T0 — señal cuasi-plana      A Autoconsumo                      53.59       215.52                    53.93       218.32                     0.63         1.30
                            B Merito diario                    53.57       215.36                    53.93       218.27                     0.66         1.35
                            C Merchant                         51.88       201.40                    52.26       204.57                     0.74         1.58
                            D Merito+Export                    53.57       215.37                    53.93       218.28                     0.66         1.35
T1 — señal horaria indexada A Autoconsumo                      54.18       220.34                    54.49       222.91                     0.58         1.17
                            B Merito diario                    54.93       226.55                    55.28       229.42                     0.63         1.27
                            C Merchant                         48.72       175.33                    49.03       177.95                     0.65         1.49
                            D Merito+Export                    54.93       226.55                    55.28       229.42                     0.63         1.27


A partir de aqui el modelo usa SOLO la demanda limpia.


## 7. Espacio de configuraciones

La batería ya **no** se construye como potencia × duración. Se define por pares
explícitos de **potencia y energía**, que son variables independientes:

| Dimensión | Valores | Sentido |
|---|---|---|
| **PV instalado** | 0 – 150 MW | Desde no instalar nada hasta ~6x la demanda pico |
| **BESS** | 25 pares MW–MWh | Potencia y energía se eligen por separado |
| **Duración** | *derivada* | `Capacidad_MWh / Potencia_MW`. **Nunca construye la malla**: se calcula a partir del par elegido y sirve para reportar, para estudiar qué geometría conviene y para consultar la curva de CAPEX |

La nueva malla **aproximadamente duplica el espacio total de diseño** respecto
a la versión anterior —116 configuraciones antes, 233 ahora— y **desacopla
explícitamente potencia y energía**.

Por qué importa: con `Capacidad = Potencia × Duración` y tres duraciones fijas,
la malla solo podía ver tres geometrías de batería. Un BESS de 20 MW podía tener
40, 80 o 120 MWh y nada más. Al liberar los dos ejes, el barrido puede
distinguir si conviene **más potencia** o **más energía**, que es una pregunta
de diseño distinta y con precios distintos.

Los 25 pares provienen de la caracterización física del cuaderno
`01B_Caracterizacion_dimensionamiento.ipynb` (§8B y §8C). Cubren ambos lados de
la rodilla de captura y conservan deliberadamente los tamaños pequeños.

**Se excluyen las duraciones mayores a 8 h.** Las curvas de CAPEX del libro
`CurvasCapexPVBESS.xlsx` están publicadas para 1 a 8 h; fuera de ese rango
`costos_curvas.capex_bess_USD()` extrapola sin avisar. Cuatro pares quedan
fuera por esa razón, no por criterio físico.

La malla sigue siendo **fina en el extremo bajo del PV**: la demanda tiene un
pico de solo 23 MW, así que el óptimo económico está en pocas decenas de MW y
una malla que empiece alto devolvería una solución de borde.

Se conservan las dos esquinas informativas:

- **PV = 0 con BESS > 0** — una batería sola contra la red, solo arbitraje.
- **PV > 0 con BESS = 0** — solo paneles, la referencia contra la que hay que
  justificar cada peso de almacenamiento.

In [14]:
# El eje de PV NO cambia. Los tamanos [30, 50, 70, 100] del cuaderno 01B eran
# puntos de caracterizacion fisica, no una malla de decision: sustituir este eje
# por ellos eliminaria PV = 0, los tamanos pequenos y los mayores de 100 MW.
PV_OPCIONES = [0, 10, 20, 30, 40, 50, 70, 100, 150]   # MW

# La bateria pasa de 'potencia x duracion' a pares explicitos potencia-energia.
# La duracion es DERIVADA: E / P. Sirve para reportar, para estudiar que
# geometria conviene y para consultar la curva de CAPEX; ya no construye nada.
#
# Las curvas CAPEX disponibles estan definidas para duraciones de 1 a 8 h. Las
# configuraciones superiores a 8 h se excluyen del analisis economico principal
# para evitar extrapolaciones no respaldadas. Los cuatro pares excluidos por esa
# razon son 10 MW/120 MWh, 20 MW/180 MWh, 20 MW/240 MWh y 30 MW/320 MWh.
PARES_MW_MWH = [
    (5, 20), (5, 40),
    (10, 20), (10, 40), (10, 80),
    (20, 20), (20, 40), (20, 80), (20, 120),
    (30, 40), (30, 80), (30, 120), (30, 180), (30, 240),
    (40, 40), (40, 80), (40, 120), (40, 180), (40, 240), (40, 320),
    (60, 80), (60, 120), (60, 180), (60, 240), (60, 320),
]

BESS_OPCIONES = [{'Potencia_MW': 0, 'Capacidad_MWh': 0, 'Duracion_h': 0}]
BESS_OPCIONES += [{'Potencia_MW': float(p), 'Capacidad_MWh': float(e),
                   'Duracion_h': e / p}
                  for p, e in PARES_MW_MWH]

DUR_CAPEX_MIN, DUR_CAPEX_MAX = 1.0, 8.0   # dominio publicado de Curva_BESS
fuera_dominio = [b for b in BESS_OPCIONES
                 if b['Potencia_MW'] > 0
                 and not (DUR_CAPEX_MIN <= b['Duracion_h'] <= DUR_CAPEX_MAX)]
assert not fuera_dominio, f'Pares fuera del dominio CAPEX: {fuera_dominio}'

configuraciones = [
    {'PV_MW': pv, **b}
    for pv in PV_OPCIONES
    for b in BESS_OPCIONES
    if not (pv == 0 and b['Potencia_MW'] == 0)
]

n_sim = sum(1 if c['Capacidad_MWh'] == 0 else len(ESTRATEGIAS)
            for c in configuraciones) * len(TARIFAS)

_durs = [b['Duracion_h'] for b in BESS_OPCIONES[1:]]
print(f'Pares BESS            : {len(PARES_MW_MWH)}  (+ el caso sin bateria)')
print(f'Duraciones derivadas  : {min(_durs):.2f} a {max(_durs):.2f} h')
print(f'Configuraciones       : {len(configuraciones)}')
print(f'Llamadas a despachar(): {n_sim:,}  (x {len(TARIFAS)} tarifas incluido)')
print(f'Tiempo estimado       : {n_sim*0.15/60:.1f} minutos')

Pares BESS            : 25  (+ el caso sin bateria)
Duraciones derivadas  : 1.00 a 8.00 h
Configuraciones       : 233
Llamadas a despachar(): 1,816  (x 2 tarifas incluido)
Tiempo estimado       : 4.5 minutos


### 7.1 Equivalencia con la malla anterior

Antes de correr la malla completa hay que comprobar que el cambio de
parametrización **no mueve ningún resultado**. Se toman configuraciones que
existían en la malla vieja y se evalúan por los dos caminos:

- **camino viejo**: `Capacidad_MWh = Potencia_MW × Duracion_h`;
- **camino nuevo**: el par `(Potencia_MW, Capacidad_MWh)` de `BESS_OPCIONES`.

Se comparan nueve magnitudes por configuración, estrategia y tarifa. La
diferencia debe ser cero o error de redondeo.

Conviene decir qué prueba esto y qué no. Como los dos caminos entregan los
**mismos flotantes** a las mismas funciones, la igualdad exacta es lo esperado:
el valor de la prueba está en detectar un par mal escrito, un argumento por
defecto cambiado o una configuración que dejó de significar lo mismo — no en
descubrir que el despacho es determinista.

In [15]:
# Configuraciones a validar. Las tres primeras y la ultima existian en la malla
# vieja (POTENCIA [0, 5, 10, 20, 40] x DURACION [2, 4, 6]); 30 MW / 120 MWh es
# NUEVA -30 MW no estaba en el eje de potencia anterior- y se incluye como
# control de que la ruta nueva produce resultados coherentes.
CASOS_EQ = [
    (20.0,  40.0, 2.0, True),
    (20.0,  80.0, 4.0, True),
    (20.0, 120.0, 6.0, True),
    (30.0, 120.0, 4.0, False),   # 30 MW no existia en la malla vieja
    (40.0,  80.0, 2.0, True),
]

METRICAS_EQ = ['Import_MWh', 'Export_MWh', 'Carga_MWh', 'Descarga_MWh',
               'Ciclos_anio', 'Beneficio_anio1_COP', 'CAPEX_COP',
               'OPEX_anual_COP', 'VPN_COP']


def evaluar_una(pv_MW, pot, cap, estrategia, c_imp):
    """Una configuracion completa, por el mismo camino que usa el barrido."""
    d = despachar(PV_PU * pv_MW, LOAD, c_imp, c_exp, estrategia,
                  power_MW=pot, cap_MWh=cap,
                  c_deg=costo_degradacion_COP_MWh(pot, cap))
    fin = evaluar_finanzas(d, c_imp, c_exp, pv_MW, pot, cap)
    return {'Import_MWh': d.Import_MW.sum(), 'Export_MWh': d.Export_MW.sum(),
            'Carga_MWh': d.Carga_MW.sum(), 'Descarga_MWh': d.Descarga_MW.sum(),
            **{k: fin[k] for k in ('Ciclos_anio', 'Beneficio_anio1_COP',
                                   'CAPEX_COP', 'OPEX_anual_COP', 'VPN_COP')}}


PV_EQ = 70.0   # un tamano intermedio del eje de PV, que no cambia
filas_eq = []

for pot, cap, dur, existia in CASOS_EQ:
    cap_viejo = pot * dur                      # como lo construia la malla vieja
    cap_nuevo = next(b['Capacidad_MWh'] for b in BESS_OPCIONES
                     if b['Potencia_MW'] == pot and b['Capacidad_MWh'] == cap)
    assert cap_viejo == cap_nuevo, (
        f'{pot:.0f} MW: la malla vieja daba {cap_viejo} MWh y la nueva {cap_nuevo}')

    for nombre_tarifa, c_imp in TARIFAS.items():
        for est in ESTRATEGIAS:
            a = evaluar_una(PV_EQ, pot, cap_viejo, est, c_imp)
            b = evaluar_una(PV_EQ, pot, cap_nuevo, est, c_imp)
            for m in METRICAS_EQ:
                escala = max(abs(a[m]), 1e-9)
                filas_eq.append({
                    'Config': f'{pot:.0f} MW / {cap:.0f} MWh',
                    'Existia antes': existia, 'Tarifa': nombre_tarifa,
                    'Estrategia': est, 'Metrica': m,
                    'Viejo': a[m], 'Nuevo': b[m],
                    'Dif_rel': abs(b[m] - a[m]) / escala,
                })

EQ = pd.DataFrame(filas_eq)

print('=' * 78)
print('EQUIVALENCIA CON LA MALLA ANTERIOR')
print('=' * 78)
print(f'Comparaciones: {len(EQ):,}  '
      f'({EQ.Config.nunique()} configuraciones x {len(TARIFAS)} tarifas '
      f'x {len(ESTRATEGIAS)} estrategias x {len(METRICAS_EQ)} metricas)')
print()

resumen_eq = (EQ.groupby(['Config', 'Existia antes']).Dif_rel.max()
                .reset_index().rename(columns={'Dif_rel': 'Dif. relativa maxima'}))
display(resumen_eq.style.format({'Dif. relativa maxima': '{:.2e}'})
                  .hide(axis='index'))

peor = EQ.Dif_rel.max()
print(f'Diferencia relativa maxima sobre {len(EQ):,} comparaciones: {peor:.2e}')
if peor == 0.0:
    print('Identidad exacta: ninguna configuracion cambia de valor.')
else:
    print('Hay diferencias no nulas. Revisar antes de continuar:')
    display(EQ.nlargest(10, 'Dif_rel'))
assert peor < 1e-12, 'La reparametrizacion cambio algun resultado'
print()
print('Nota: 30 MW / 120 MWh no existia en la malla vieja. Su fila compara la')
print('construccion 30 x 4 h contra el par (30, 120), que es la misma bateria.')

EQUIVALENCIA CON LA MALLA ANTERIOR
Comparaciones: 360  (5 configuraciones x 2 tarifas x 4 estrategias x 9 metricas)



Config,Existia antes,Dif. relativa maxima
20 MW / 120 MWh,True,0.00e+00
20 MW / 40 MWh,True,0.00e+00
20 MW / 80 MWh,True,0.00e+00
30 MW / 120 MWh,False,0.00e+00
40 MW / 80 MWh,True,0.00e+00


Diferencia relativa maxima sobre 360 comparaciones: 0.00e+00
Identidad exacta: ninguna configuracion cambia de valor.

Nota: 30 MW / 120 MWh no existia en la malla vieja. Su fila compara la
construccion 30 x 4 h contra el par (30, 120), que es la misma bateria.


## 8. Barrido

Cada combinación de configuración × estrategia × tarifa se simula durante un año
completo, se valida físicamente y se evalúa financieramente. Las configuraciones
sin batería se simulan una sola vez y su resultado se replica para las cuatro
estrategias, ya que sin almacenamiento no hay decisión de operación que tomar.

In [16]:
import time
import motor_multianual as MM

# Anclas del algoritmo adaptativo. Los segmentos donde el EFC proyectado alcanza
# la envolvente, y los años en que la edad cruza la vida calendario, se simulan
# explicitamente: el evento se localiza hora a hora, nunca por interpolacion.
ANIOS_ANCLA = [1, 5, 10, 15, 20]

filas = []
fallos = []
t_inicio = time.time()

for k, cfg in enumerate(configuraciones, 1):
    cap, pot, pv = cfg['Capacidad_MWh'], cfg['Potencia_MW'], cfg['PV_MW']

    # Precio sombra sobre R1: al agotar la envolvente se renueva el bloque de
    # almacenamiento, no el EPC completo, asi que es R1 lo que hay que amortizar.
    c_deg = costos.costo_degradacion_R1_COP_MWh(pot, cap, DoD, ETA_RT)
    capex0 = costos.capex_pv(pv) + costos.capex_bess(pot, cap)
    opex = costos.opex_pv_anual(pv) + costos.opex_bess_anual(cap, power_MW=pot)

    for nombre_tarifa, c_imp in TARIFAS.items():
        ests = ESTRATEGIAS if cap > 0 else ESTRATEGIAS[:1]
        resultados_cfg = {}

        for est in ests:
            r = MM.simular_horizonte(
                despachar, PV_PU, LOAD, c_imp, c_exp, est,
                pv, pot, cap, DoD=DoD, eff_rt=ETA_RT, c_deg_COP_MWh=c_deg,
                capex_inicial_COP=capex0, opex_anual_COP=opex,
                anios_ancla=ANIOS_ANCLA)

            # La validacion fisica se hace sobre el despacho del ano 1, que es
            # el unico que el motor garantiza haber simulado explicitamente.
            d1 = despachar(PV_PU * pv, LOAD, c_imp, c_exp, est,
                           power_MW=pot, cap_MWh=cap, c_deg=c_deg)
            problemas = validar(d1, pot, cap)
            if problemas:
                fallos.append({**cfg, 'Tarifa': nombre_tarifa,
                               'Estrategia': est, 'Problemas': problemas})

            a1 = r['anio1']
            resultados_cfg[est] = {
                **cfg, 'Tarifa': nombre_tarifa, 'Estrategia': est,
                'VPN_COP': r['VPN_COP'], 'TIR': r['TIR'],
                'Payback_anios': r['Payback_anios'],
                'CAPEX_COP': capex0, 'OPEX_anual_COP': opex,
                'Beneficio_anio1_COP': r['Beneficio_anio1_COP'],
                'EFC_anio1': r['EFC_anio1'],
                'Ciclos_anio': r['EFC_anio1'],      # nombre historico, mismo dato
                'EFC_total_horizonte': r['EFC_total_descargado'],
                'EFC_bloque_final': r['EFC_bloque_activo_final'],
                'Edad_bloque_final': r['Edad_BESS_final'],
                'N_renovaciones': r['n_renovaciones'],
                'Anios_renovacion': ';'.join(str(a) for a in r['Anios_renovacion']),
                'Costo_renovaciones_VP_COP': r['Costo_renovaciones_VP_COP'],
                'Costo_renovaciones_nominal_COP': r['Costo_renovaciones_nominal_COP'],
                'Importacion_anio1': a1['Importacion_MWh'],
                'Exportacion_anio1': a1['Exportacion_MWh'],
                'Carga_BESS_anio1': a1['Carga_BESS_MWh'],
                'Descarga_BESS_anio1': a1['Descarga_BESS_MWh'],
                'Vertimiento_anio1': a1['Vertimiento_MWh'],
                'Autoconsumo_%': a1['Autoconsumo_pct'],
                'N_despachos_multianuales': r['n_despachos'],
            }

        if cap == 0:   # sin bateria no hay decision operativa: se replica
            base = resultados_cfg[ESTRATEGIAS[0]]
            for est in ESTRATEGIAS:
                filas.append({**base, 'Estrategia': est})
        else:
            filas.extend(resultados_cfg.values())

    if k % 10 == 0 or k == len(configuraciones):
        print(f'  {k:>3}/{len(configuraciones)} configuraciones  '
              f'({time.time()-t_inicio:5.1f}s)')

R = pd.DataFrame(filas)
R['VPN_mil_MM'] = R.VPN_COP / 1e9
R['CAPEX_mil_MM'] = R.CAPEX_COP / 1e9

_t_total = time.time() - t_inicio
print(f'\nListo en {_t_total:.1f}s — {len(R):,} evaluaciones')
print(f'Validaciones fallidas: {len(fallos)}')

_d = R.N_despachos_multianuales
print()
print('COSTO REAL DEL ALGORITMO ADAPTATIVO')
print(f'  Llamadas a despachar() del motor : {_d.sum():,}')
print(f'  Despachos por evaluacion         : media {_d.mean():.2f} | '
      f'min {_d.min()} | P50 {_d.median():.0f} | P90 {_d.quantile(.90):.0f} | max {_d.max()}')
print(f'  Frente a 20 anos explicitos      : {_d.sum()/(len(R)*20):.1%} del trabajo')

SALIDA = Path('resultados_nb02')
SALIDA.mkdir(exist_ok=True)
R.to_csv(SALIDA / 'C3B_multianual.csv', index=False)
print(f"\nGuardado: {SALIDA / 'C3B_multianual.csv'}  (C1A no se sobrescribe)")
if fallos:
    display(pd.DataFrame(fallos).head(20))

   10/233 configuraciones  (108.8s)


   20/233 configuraciones  (214.4s)


   30/233 configuraciones  (303.4s)


   40/233 configuraciones  (374.0s)


   50/233 configuraciones  (448.2s)


...

  200/233 configuraciones  (1754.9s)


  210/233 configuraciones  (1841.1s)


  220/233 configuraciones  (1937.0s)


  230/233 configuraciones  (2029.0s)


  233/233 configuraciones  (2054.9s)

Listo en 2055.0s — 1,864 evaluaciones
Validaciones fallidas: 0

COSTO REAL DEL ALGORITMO ADAPTATIVO
  Llamadas a despachar() del motor : 9,516
  Despachos por evaluacion         : media 5.11 | min 5 | P50 5 | P90 5 | max 9
  Frente a 20 anos explicitos      : 25.5% del trabajo

Guardado: resultados_nb02\C3B_multianual.csv  (C1A no se sobrescribe)


## 9. Resultados

### 9.1 Las mejores configuraciones

Se ordena todo el espacio por VPN y se muestran las diez mejores combinaciones de
cada tarifa. La pregunta que responde esta tabla es directamente la que planteó el
notebook: **cuánto PV, cuánto BESS y con qué regla**.

In [17]:
COLS = ['PV_MW', 'Potencia_MW', 'Duracion_h', 'Estrategia', 'VPN_mil_MM',
        'TIR', 'Payback_anios', 'CAPEX_mil_MM',
        'Autoconsumo_%', 'Ciclos_anio']   # C1.4: sin costo nivelado

for nombre_tarifa in TARIFAS:
    sub = R[R.Tarifa == nombre_tarifa].nlargest(10, 'VPN_mil_MM')
    print('=' * 96)
    print(f'MEJORES CONFIGURACIONES — {nombre_tarifa}')
    print('=' * 96)
    display(sub[COLS].style.format({
        'VPN_mil_MM': '{:,.1f}', 'TIR': '{:.1%}', 'Payback_anios': '{:,.0f}',
        'CAPEX_mil_MM': '{:,.1f}',
        'Autoconsumo_%': '{:.1f}', 'Ciclos_anio': '{:.0f}'}).hide(axis='index'))

MEJORES CONFIGURACIONES — T0 — señal cuasi-plana


PV_MW,Potencia_MW,Duracion_h,Estrategia,VPN_mil_MM,TIR,Payback_anios,CAPEX_mil_MM,Autoconsumo_%,Ciclos_anio
150,40.000000,8.000000,B Merito diario,315.5,19.0%,8,479.4,89.4,300
150,40.000000,8.000000,A Autoconsumo,315.5,19.0%,8,479.4,89.4,300
150,60.000000,5.333333,B Merito diario,312.2,18.6%,8,496.4,91.2,311
150,60.000000,5.333333,A Autoconsumo,312.2,18.6%,8,496.4,91.2,311
150,40.000000,8.000000,D Merito+Export,310.4,18.8%,8,479.4,88.5,309
150,40.000000,6.000000,B Merito diario,302.2,19.2%,8,446.4,80.9,330
150,40.000000,6.000000,A Autoconsumo,302.1,19.2%,8,446.4,80.9,330
150,30.000000,8.000000,B Merito diario,300.7,19.3%,8,437.9,79.8,320
150,30.000000,8.000000,A Autoconsumo,300.6,19.3%,8,437.9,79.8,320
150,60.000000,5.333333,D Merito+Export,300.3,18.3%,8,496.4,89.2,336


MEJORES CONFIGURACIONES — T1 — señal horaria indexada


PV_MW,Potencia_MW,Duracion_h,Estrategia,VPN_mil_MM,TIR,Payback_anios,CAPEX_mil_MM,Autoconsumo_%,Ciclos_anio
150,40.000000,8.000000,B Merito diario,325.0,19.2%,8,479.4,89.4,300
150,40.000000,8.000000,A Autoconsumo,324.1,19.2%,8,479.4,89.4,300
150,60.000000,5.333333,B Merito diario,319.9,18.8%,8,496.4,91.2,311
150,40.000000,8.000000,D Merito+Export,319.8,19.1%,8,479.4,88.5,309
150,60.000000,5.333333,A Autoconsumo,319.1,18.8%,8,496.4,91.2,311
150,40.000000,6.000000,B Merito diario,314.0,19.5%,8,446.4,80.8,329
150,30.000000,8.000000,B Merito diario,313.7,19.7%,8,437.9,79.7,319
150,40.000000,6.000000,A Autoconsumo,313.6,19.5%,8,446.4,80.9,330
150,30.000000,8.000000,A Autoconsumo,313.0,19.7%,8,437.9,79.8,320
150,30.000000,8.000000,D Merito+Export,312.6,19.7%,8,437.9,79.5,322


In [18]:
# La configuración ganadora de cada tarifa, en detalle
print('=' * 78)
print('MEJOR CONFIGURACIÓN ENCONTRADA EN LA MALLA — POR TARIFA')
print('=' * 78)

for nombre_tarifa in TARIFAS:
    mejor = R[R.Tarifa == nombre_tarifa].nlargest(1, 'VPN_mil_MM').iloc[0]
    tarifa_actual = (TARIFAS[nombre_tarifa] * demanda).sum() / demanda.sum() / 1000
    print(f'\n{nombre_tarifa}')
    print(f'  PV                    : {mejor.PV_MW:,.0f} MW')
    print(f'  BESS                  : {mejor.Potencia_MW:,.0f} MW / '
          f'{mejor.Capacidad_MWh:,.0f} MWh ({mejor.Duracion_h:.1f} h)')
    print(f'  Estrategia            : {mejor.Estrategia}')
    print(f'  VPN                   : {mejor.VPN_mil_MM:,.1f} mil millones COP')
    print(f'  TIR                   : {mejor.TIR:.1%}   (WACC {WACC:.0%})')
    print(f'  Payback descontado    : {mejor.Payback_anios:,.0f} años')
    print(f'  CAPEX                 : {mejor.CAPEX_mil_MM:,.1f} mil millones COP')
    print(f'  Costo nivelado        : NO REPORTAR — pendiente de modelo multianual '
          f'(tarifa actual {tarifa_actual:,.1f} COP/kWh)')
    print(f'  Autoconsumo           : {mejor["Autoconsumo_%"]:.1f} % de la demanda')

print()
print('Es la mejor de las 233 configuraciones evaluadas, operada con una regla')
print('heurística. No es un despacho matemáticamente óptimo: este cuaderno no')
print('resuelve ningún problema de optimización de operación.')

MEJOR CONFIGURACIÓN ENCONTRADA EN LA MALLA — POR TARIFA

T0 — señal cuasi-plana
  PV                    : 150 MW
  BESS                  : 40 MW / 320 MWh (8.0 h)
  Estrategia            : B Merito diario
  VPN                   : 315.5 mil millones COP
  TIR                   : 19.0%   (WACC 10%)
  Payback descontado    : 8 años
  CAPEX                 : 479.4 mil millones COP
  Costo nivelado        : NO REPORTAR — pendiente de modelo multianual (tarifa actual 657.4 COP/kWh)
  Autoconsumo           : 89.4 % de la demanda

T1 — señal horaria indexada
  PV                    : 150 MW
  BESS                  : 40 MW / 320 MWh (8.0 h)
  Estrategia            : B Merito diario
  VPN                   : 325.0 mil millones COP
  TIR                   : 19.2%   (WACC 10%)
  Payback descontado    : 8 años
  CAPEX                 : 479.4 mil millones COP
  Costo nivelado        : NO REPORTAR — pendiente de modelo multianual (tarifa actual 657.4 COP/kWh)
  Autoconsumo           : 89.4 % de la d

### 9.2 ¿Cuánto PV conviene instalar?

Esta es la decisión de mayor impacto de todo el proyecto, así que conviene verla
sola y sin adornos.

La gráfica recorre todos los tamaños de PV evaluados y, para cada uno, dibuja dos
curvas:

- **Solo PV** — el VPN si se instalan paneles sin batería.
- **PV + BESS** — el VPN de la *mejor* combinación de batería y estrategia
  disponible para ese tamaño de PV.

**Cómo leerla:**

| Qué mirar | Qué significa |
|---|---|
| El **punto más alto** de la curva azul | El tamaño óptimo de PV. Esa es la respuesta. |
| La **subida** hasta el máximo | Cada MW adicional desplaza compras de energía a la red: se paga solo. |
| La **bajada** después del máximo | El PV ya cubre toda la demanda que puede; lo que sobra hay que exportarlo a precio de bolsa, que paga mucho menos de lo que ahorra el autoconsumo. Por eso el VPN cae. |
| Dónde **cruza el cero** | A partir de ahí el proyecto no se paga. |
| La **distancia entre las dos curvas** | Lo que cuesta añadir almacenamiento. Si la naranja va por debajo de la azul, la batería resta valor a los precios supuestos. |

La forma de campana no es casualidad: es el resultado de que el **beneficio crece
de forma cóncava** (cada MW extra de sol vale menos que el anterior, porque la
demanda barata ya está cubierta) mientras que el **CAPEX crece de forma lineal**.
El óptimo está donde las dos pendientes se igualan.

Debajo de la gráfica, una tabla por tarifa con las mismas cifras más el detalle
que la curva no puede mostrar: **qué batería concreta** (potencia, capacidad,
duración y estrategia) es la mejor para cada tamaño de PV, y cuánto suma o resta
frente a no instalarla. Es la comparación que permite decidir con números y no
solo con la forma de la curva.

In [19]:
# ================================================================
# ¿Cuánto PV conviene instalar?
# ----------------------------------------------------------------
# Se dibuja un panel por tarifa. Dentro de cada panel hay dos
# curvas que recorren todos los tamaños de PV evaluados:
#   - "Solo PV"    : el proyecto sin batería
#   - "PV + BESS"  : el proyecto con la MEJOR batería disponible
#                    para ese mismo tamaño de PV
# La distancia entre ambas es el aporte (o el costo) del
# almacenamiento.
# ================================================================

fig, axes = plt.subplots(1, len(TARIFAS), figsize=(13.5, 4.9), sharey=True)

tablas_comparacion = {}   # se llena en el bucle y se muestra al final

for ax, nombre_tarifa in zip(axes, TARIFAS):

    # ---- 1. Resultados de esta tarifa --------------------------
    sub = R[R.Tarifa == nombre_tarifa]

    # ---- 2. Curva "solo PV" ------------------------------------
    # Para cada tamaño de PV existe una sola configuración sin
    # batería, así que el max() solo sirve para colapsar el grupo.
    solo_pv = sub[sub.Capacidad_MWh == 0].groupby('PV_MW').VPN_mil_MM.max()

    # No invertir nada vale exactamente cero. Añadir ese punto
    # ancla la curva en el origen y deja ver desde qué tamaño el
    # proyecto empieza a crear valor.
    solo_pv.loc[0] = 0.0
    solo_pv = solo_pv.sort_index()

    # ---- 3. Curva "PV + BESS" ----------------------------------
    # Para cada tamaño de PV hay muchas configuraciones de batería
    # (potencia x duración x estrategia). Interesa la mejor de
    # todas. Se usa idxmax en lugar de max porque necesitamos la
    # FILA COMPLETA de la ganadora: así podemos reportar después
    # de qué batería y qué estrategia se trata, no solo su VPN.
    con_bess = sub[sub.Capacidad_MWh > 0]
    ganadora = (con_bess.loc[con_bess.groupby('PV_MW').VPN_mil_MM.idxmax()]
                        .set_index('PV_MW').sort_index())

    # ---- 4. Dibujar --------------------------------------------
    # El área sombreada entre las dos curvas hace visible de un
    # vistazo si el almacenamiento suma o resta.
    ax.fill_between(ganadora.index,
                    ganadora.VPN_mil_MM.values,
                    solo_pv.reindex(ganadora.index).values,
                    color='#eb6834', alpha=0.10, zorder=1)

    ax.plot(ganadora.index, ganadora.VPN_mil_MM.values,
            marker='o', markersize=6, linewidth=2.2,
            color='#eb6834', label='PV + BESS (mejor caso)', zorder=3)
    ax.plot(solo_pv.index, solo_pv.values,
            marker='o', markersize=6, linewidth=2.2,
            color='#2a78d6', label='Solo PV', zorder=4)

    # Línea de rentabilidad: por debajo, el proyecto destruye valor.
    ax.axhline(0, color=TINTA_SUAVE, linewidth=1.2, zorder=2)

    # ---- 5. Señalar el óptimo ----------------------------------
    # Etiqueta directa sobre el punto, para no obligar a rastrear
    # el valor en los ejes.
    pv_opt = solo_pv.idxmax()
    vpn_opt = solo_pv.max()
    ax.scatter([pv_opt], [vpn_opt], s=150, facecolors='none',
               edgecolors=TINTA, linewidths=2, zorder=5)
    ax.annotate(f'óptimo: {pv_opt:,.0f} MW\n{vpn_opt:,.0f} mil MM COP',
                (pv_opt, vpn_opt), textcoords='offset points',
                xytext=(10, 6), fontsize=9, fontweight='bold',
                color=TINTA, zorder=6)

    ax.set_xlabel('PV instalado [MW]')
    ax.set_title(nombre_tarifa, loc='left', fontweight='bold')
    ax.grid(axis='x', visible=False)

    # ---- 6. Tabla de respaldo ----------------------------------
    # La curva muestra el "cuánto", pero no el "cuál": para cada
    # tamaño de PV se detalla qué batería resultó ganadora.
    tabla = pd.DataFrame(index=solo_pv.index)
    tabla.index.name = 'PV [MW]'
    tabla['VPN solo PV'] = solo_pv

    tabla['Mejor BESS'] = ganadora.apply(
        lambda f: f'{f.Potencia_MW:,.0f} MW / {f.Capacidad_MWh:,.0f} MWh '
                  f'({f.Duracion_h:,.1f} h)', axis=1)
    tabla['Estrategia'] = ganadora.Estrategia
    tabla['VPN con BESS'] = ganadora.VPN_mil_MM

    # La columna decisiva: cuánto cambia el VPN por añadir la batería.
    tabla['Aporte del BESS'] = tabla['VPN con BESS'] - tabla['VPN solo PV']

    tablas_comparacion[nombre_tarifa] = tabla

axes[0].set_ylabel('VPN [mil millones COP]')
axes[0].legend(frameon=False, fontsize=9, loc='lower left')
fig.suptitle('¿Cuánto PV conviene instalar? El punto más alto de la curva azul es la respuesta',
             x=0.055, ha='left', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()


# ================================================================
# Tablas de respaldo, una por tarifa
# ----------------------------------------------------------------
# Se resalta en verde el mejor VPN de cada columna, para localizar
# de inmediato la fila óptima sin tener que recorrer los números.
# ================================================================

for nombre_tarifa, tabla in tablas_comparacion.items():
    display(
        tabla.style
             .format({'VPN solo PV': '{:,.1f}',
                      'VPN con BESS': '{:,.1f}',
                      'Aporte del BESS': '{:+,.1f}'}, na_rep='—')
             .highlight_max(subset=['VPN solo PV', 'VPN con BESS'],
                            color='#d8f0e4')
             .set_caption(f'<b>{nombre_tarifa}</b> — VPN [mil millones COP] '
                          f'y batería óptima para cada tamaño de PV')
    )

print('La columna "Aporte del BESS" es la que decide: si es negativa en toda la')
print('tabla, el almacenamiento no entra en el portafolio a los precios supuestos.')

,VPN solo PV,Mejor BESS,Estrategia,VPN con BESS,Aporte del BESS
PV [MW],,,,,
0,0.0,5 MW / 20 MWh (4.0 h),A Autoconsumo,-13.8,-13.8
10,45.7,5 MW / 20 MWh (4.0 h),B Merito diario,32.1,-13.5
20,93.7,5 MW / 20 MWh (4.0 h),A Autoconsumo,80.5,-13.2
30,136.9,5 MW / 20 MWh (4.0 h),A Autoconsumo,128.3,-8.6
40,162.8,5 MW / 20 MWh (4.0 h),A Autoconsumo,164.1,+1.4
50,176.8,10 MW / 40 MWh (4.0 h),A Autoconsumo,187.9,+11.1
70,189.7,20 MW / 80 MWh (4.0 h),A Autoconsumo,225.0,+35.3
100,193.9,40 MW / 240 MWh (6.0 h),A Autoconsumo,264.1,+70.2
150,186.2,40 MW / 320 MWh (8.0 h),B Merito diario,315.5,+129.3


,VPN solo PV,Mejor BESS,Estrategia,VPN con BESS,Aporte del BESS
PV [MW],,,,,
0,0.0,5 MW / 20 MWh (4.0 h),C Merchant,-11.7,-11.7
10,43.8,5 MW / 20 MWh (4.0 h),C Merchant,32.3,-11.5
20,90.0,5 MW / 20 MWh (4.0 h),C Merchant,78.7,-11.3
30,131.3,5 MW / 20 MWh (4.0 h),B Merito diario,124.5,-6.8
40,155.4,10 MW / 20 MWh (2.0 h),B Merito diario,162.4,+6.9
50,168.3,20 MW / 80 MWh (4.0 h),B Merito diario,193.8,+25.5
70,179.6,20 MW / 80 MWh (4.0 h),B Merito diario,236.3,+56.7
100,182.5,40 MW / 240 MWh (6.0 h),B Merito diario,282.4,+99.9
150,173.7,40 MW / 320 MWh (8.0 h),B Merito diario,325.0,+151.4


La columna "Aporte del BESS" es la que decide: si es negativa en toda la
tabla, el almacenamiento no entra en el portafolio a los precios supuestos.


### 9.4 ¿Cuánto importa la estrategia frente al dimensionamiento?

Esta es la pregunta central del notebook. Se descompone la variabilidad del VPN en
dos fuentes:

- **Dimensionamiento:** cuánto cambia el VPN al moverse entre configuraciones
- **Estrategia:** cuánto cambia el VPN dentro de una misma configuración, al
  cambiar solo la regla de operación

Si la primera domina a la segunda, la conclusión práctica es que **primero hay que
acertar el tamaño** y solo después afinar la operación.

In [20]:
comparacion = []
for nombre_tarifa in TARIFAS:
    sub = R[(R.Tarifa == nombre_tarifa) & (R.Capacidad_MWh > 0)]

    rango_dimension = sub.VPN_mil_MM.max() - sub.VPN_mil_MM.min()
    rango_estrategia = (sub.groupby(['PV_MW', 'Potencia_MW', 'Capacidad_MWh'])
                          .VPN_mil_MM.agg(lambda s: s.max() - s.min()))

    comparacion.append({
        'Tarifa': nombre_tarifa,
        'Rango por dimensionamiento': rango_dimension,
        'Rango por estrategia (mediana)': rango_estrategia.median(),
        'Rango por estrategia (máx)': rango_estrategia.max(),
        'Razón dimensión/estrategia': rango_dimension / max(rango_estrategia.median(), 1e-9),
    })

display(pd.DataFrame(comparacion).style.format({
    'Rango por dimensionamiento': '{:,.1f}',
    'Rango por estrategia (mediana)': '{:,.1f}',
    'Rango por estrategia (máx)': '{:,.1f}',
    'Razón dimensión/estrategia': '{:,.0f}x'}).hide(axis='index'))

print('\nTodas las cifras en mil millones de COP de VPN.')

Tarifa,Rango por dimensionamiento,Rango por estrategia (mediana),Rango por estrategia (máx),Razón dimensión/estrategia
T0 — señal cuasi-plana,537.3,6.8,222.4,79x
T1 — señal horaria indexada,535.8,25.3,248.6,21x



Todas las cifras en mil millones de COP de VPN.


In [21]:
# Ranking de estrategias por tarifa: VPN medio sobre todas las configuraciones con BESS
fig, axes = plt.subplots(1, len(TARIFAS), figsize=(13.5, 4.6), sharey=True)

for ax, nombre_tarifa in zip(axes, TARIFAS):
    sub = R[(R.Tarifa == nombre_tarifa) & (R.Capacidad_MWh > 0)]
    medio = sub.groupby('Estrategia').VPN_mil_MM.mean().reindex(ESTRATEGIAS)

    barras = ax.bar(range(len(medio)), medio.values,
                    color=[COLOR_ESTRATEGIA[e] for e in medio.index],
                    width=0.62, zorder=3)
    for i, (nombre, valor) in enumerate(medio.items()):
        ax.annotate(f'{valor:,.0f}', (i, valor), ha='center',
                    va='bottom' if valor >= 0 else 'top',
                    xytext=(0, 4 if valor >= 0 else -4),
                    textcoords='offset points', fontsize=9, color=TINTA)

    ax.set_xticks(range(len(medio)))
    ax.set_xticklabels([e.split(' ', 1)[0] for e in medio.index])
    ax.axhline(0, color=TINTA_SUAVE, linewidth=1)
    ax.set_title(nombre_tarifa, loc='left', fontweight='bold')
    ax.grid(axis='x', visible=False)

axes[0].set_ylabel('VPN medio [mil millones COP]')

# Leyenda con los nombres completos, ya que el eje usa abreviaturas
manijas = [plt.Rectangle((0, 0), 1, 1, color=COLOR_ESTRATEGIA[e]) for e in ESTRATEGIAS]
fig.legend(manijas, ESTRATEGIAS, frameon=False, ncol=4,
           loc='lower center', bbox_to_anchor=(0.5, -0.06))
fig.suptitle('VPN medio por estrategia, sobre todas las configuraciones con BESS',
             x=0.06, ha='left', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

### 9.5 Valor marginal del almacenamiento

Una cosa es que una configuración tenga buen VPN y otra distinta es que *añadir
más batería* mejore el resultado. La curva siguiente fija el PV en su valor óptimo
y recorre la capacidad de BESS, mostrando el VPN incremental frente a no tener
batería.

Hay tres lecturas posibles y las tres son informativas:

- Si la curva **sube y luego baja**, su máximo marca el tamaño óptimo de batería.
- Si la curva **baja desde el primer punto**, el almacenamiento no se paga a los
  precios supuestos: la respuesta correcta es instalar solo PV.
- Si la curva **sube en todo el rango**, la malla se quedó corta y hay que extenderla.

In [22]:
fig, axes = plt.subplots(1, len(TARIFAS), figsize=(14, 4.8), sharey=True)

for ax, nombre_tarifa in zip(axes, TARIFAS):
    sub = R[R.Tarifa == nombre_tarifa]
    pv_optimo = sub.nlargest(1, 'VPN_mil_MM').iloc[0].PV_MW

    base = sub[(sub.PV_MW == pv_optimo) & (sub.Capacidad_MWh == 0)].VPN_mil_MM.max()
    linea = sub[sub.PV_MW == pv_optimo]

    for est in ESTRATEGIAS:
        d = (linea[linea.Estrategia == est]
             .groupby('Capacidad_MWh').VPN_mil_MM.max().sort_index())
        ax.plot(d.index, d.values - base, marker='o', markersize=5,
                linewidth=2, color=COLOR_ESTRATEGIA[est], label=est, zorder=3)

    ax.axhline(0, color=TINTA_SUAVE, linewidth=1.2)
    ax.set_title(f'{nombre_tarifa} — PV fijo en {pv_optimo:,.0f} MW',
                 loc='left', fontweight='bold')
    ax.set_xlabel('Capacidad BESS [MWh]')

axes[0].set_ylabel('VPN incremental vs. sin BESS\n[mil millones COP]')
axes[0].legend(frameon=False, fontsize=9)
fig.suptitle('¿Cuánto almacenamiento vale la pena añadir?',
             x=0.06, ha='left', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

### 9.6 ¿A qué precio entra el almacenamiento en el portafolio?

Si el barrido concluye que no conviene instalar batería, la pregunta útil no es
*"¿cuánta batería?"* sino **"¿cuánto tendría que bajar su precio para que entre?"**.
Esa cifra es un resultado accionable: se compara contra las curvas de aprendizaje
de la tecnología y dice en qué año la decisión cambiaría.

El procedimiento fija el PV en su valor óptimo y baja el CAPEX del componente de
energía del BESS hasta que alguna configuración con batería supera en VPN a la
configuración de solo PV.

> **Nota:** el despacho se mantiene fijo mientras se varía el CAPEX. En rigor un
> CAPEX menor reduce el precio sombra de degradación y haría ciclar más la
> batería, mejorando su desempeño. Por eso la cifra que se obtiene es
> **conservadora**: el precio real de entrada es algo más alto.

> **Corregido en C1.3.** Esta sensibilidad reconstruía antes el proyecto con
> constantes planas —850 USD/kWp, 220 USD/kWh, 120 USD/kW— que producían un
> CAPEX un 27,5 % mayor que el del análisis principal, así que el umbral que
> reportaba no era comparable con el modelo. Ahora usa el mismo motor de CAPEX:
> `costos.capex_pv()` para el PV y toda la estructura de `costos_curvas` para el
> BESS, escalada por el precio que se sensibiliza. La versión anterior queda
> **reemplazada**, no corregida en sus cifras: los umbrales publicados antes de
> C1.3 no son comparables con estos.

In [23]:
# ------------------------------------------------------------------
# NO REPORTAR — pendiente de reconstruccion sobre el motor multianual
# ------------------------------------------------------------------
# Esta celda reconstruia su propio flujo de caja con la proyeccion de un solo
# ano, `beneficio_anio1 * (1 - DEGRADACION_PV)**(t-1)`, y con una unica
# reposicion. Desde C3B el resultado principal sale de
# `motor_multianual.simular_horizonte`, que despacha cada ano por separado y
# admite tantas renovaciones como impongan los contadores.
#
# Dejarla operativa produciria cifras internamente coherentes pero en
# contradiccion con el motor. Se desactiva hasta reconstruirla sobre
# `simular_horizonte`, igual que se hizo en C1.4 con el costo nivelado.
print('PRECIO DE ENTRADA DEL ALMACENAMIENTO')
print('NO REPORTAR — pendiente de reconstruccion sobre el motor multianual')

PRECIO DE ENTRADA DEL ALMACENAMIENTO
NO REPORTAR — pendiente de reconstruccion sobre el motor multianual


## 10. Análisis de sensibilidad

El VPN depende de supuestos que no controlamos: el precio de las baterías, el
precio de la energía y el costo del capital. Esta sección mide **cuánto de la
conclusión sobrevive** si esos supuestos se mueven.

Se re-evalúa el flujo de caja de la configuración óptima variando un parámetro a
la vez, sin volver a simular el despacho (el despacho físico no cambia con el
CAPEX ni con el WACC).

> El precio de la energía sí cambiaría el despacho, así que su sensibilidad se
> reporta como una elasticidad aproximada y se marca como tal.

In [24]:
# ------------------------------------------------------------------
# NO REPORTAR — pendiente de reconstruccion sobre el motor multianual
# ------------------------------------------------------------------
# Esta celda reconstruia su propio flujo de caja con la proyeccion de un solo
# ano, `beneficio_anio1 * (1 - DEGRADACION_PV)**(t-1)`, y con una unica
# reposicion. Desde C3B el resultado principal sale de
# `motor_multianual.simular_horizonte`, que despacha cada ano por separado y
# admite tantas renovaciones como impongan los contadores.
#
# Dejarla operativa produciria cifras internamente coherentes pero en
# contradiccion con el motor. Se desactiva hasta reconstruirla sobre
# `simular_horizonte`, igual que se hizo en C1.4 con el costo nivelado.
print('SENSIBILIDAD DEL VPN DE LA CONFIGURACION')
print('NO REPORTAR — pendiente de reconstruccion sobre el motor multianual')

SENSIBILIDAD DEL VPN DE LA CONFIGURACION
NO REPORTAR — pendiente de reconstruccion sobre el motor multianual


In [25]:
# ------------------------------------------------------------------
# NO REPORTAR — pendiente de reconstruccion sobre el motor multianual
# ------------------------------------------------------------------
# Esta celda reconstruia su propio flujo de caja con la proyeccion de un solo
# ano, `beneficio_anio1 * (1 - DEGRADACION_PV)**(t-1)`, y con una unica
# reposicion. Desde C3B el resultado principal sale de
# `motor_multianual.simular_horizonte`, que despacha cada ano por separado y
# admite tantas renovaciones como impongan los contadores.
#
# Dejarla operativa produciria cifras internamente coherentes pero en
# contradiccion con el motor. Se desactiva hasta reconstruirla sobre
# `simular_horizonte`, igual que se hizo en C1.4 con el costo nivelado.
print('PUNTOS DE QUIEBRE')
print('NO REPORTAR — pendiente de reconstruccion sobre el motor multianual')

PUNTOS DE QUIEBRE
NO REPORTAR — pendiente de reconstruccion sobre el motor multianual


In [26]:
# ------------------------------------------------------------------
# NO REPORTAR — pendiente de reconstruccion sobre el motor multianual
# ------------------------------------------------------------------
# Esta celda reconstruia su propio flujo de caja con la proyeccion de un solo
# ano, `beneficio_anio1 * (1 - DEGRADACION_PV)**(t-1)`, y con una unica
# reposicion. Desde C3B el resultado principal sale de
# `motor_multianual.simular_horizonte`, que despacha cada ano por separado y
# admite tantas renovaciones como impongan los contadores.
#
# Dejarla operativa produciria cifras internamente coherentes pero en
# contradiccion con el motor. Se desactiva hasta reconstruirla sobre
# `simular_horizonte`, igual que se hizo en C1.4 con el costo nivelado.
print('PRUEBA DE LA RAMA DE REPOSICION')
print('NO REPORTAR — pendiente de reconstruccion sobre el motor multianual')

PRUEBA DE LA RAMA DE REPOSICION
NO REPORTAR — pendiente de reconstruccion sobre el motor multianual


## 10.4 Evolución del dimensionamiento con las correcciones de datos

*Comprobación opcional. Está desactivada por defecto.*

Dos correcciones independientes tocaron las entradas físicas del modelo: el
recurso solar pasó de `Oriente.xlsx` al TMY, y la demanda pasó de cruda a
limpia. Ninguna de las dos tocó la economía.

La pregunta era si el dimensionamiento **sobrevive a las dos**. Para responderla
hay que repetir la búsqueda completa con cada combinación de entradas: comparar
solo el VPN de una configuración fija no bastaría, porque lo que se quiere saber
es si *otra* configuración pasa a ganar.

Eso cuesta tres barridos en vez de uno, unos 10 minutos adicionales. Como la
prueba ya cumplió su función y las entradas quedaron congeladas, se conserva
**detrás de una bandera** en lugar de correrla siempre:

```python
RUN_VALIDACION_HISTORICA = False   # declarada arriba, con los demás parámetros
```

Con `True` reproduce los tres estados —A `Oriente.xlsx` + demanda cruda,
B TMY + demanda cruda, C TMY + demanda limpia— usando el mismo motor de
simulación, sin duplicarlo. Las series entran por argumento y se copian, y al
terminar se comprueba que los vectores globales siguen intactos: el modelo
principal siempre acaba usando TMY 2025 y la demanda de `demanda.limpiar()`.

El paso C es el mismo estado que el barrido principal, así que además sirve de
comprobación cruzada: si `mejor_por_tarifa()` no reprodujera el resultado de la
celda 8, la validación fallaría.

In [27]:
def mejor_por_tarifa(pv_pu, load):
    """Repite la busqueda completa y devuelve la ganadora de cada tarifa.

    Misma malla, mismas estrategias, mismos supuestos economicos: lo unico que
    cambia son las dos series de entrada, que llegan por argumento. Replica la
    regla del barrido principal de simular una sola estrategia cuando no hay
    bateria. No lee ni escribe PV_PU ni LOAD.
    """
    salida = {}
    for nombre_tarifa, c_imp in TARIFAS.items():
        mejor = None
        for cfg in configuraciones:
            pot, cap = cfg['Potencia_MW'], cfg['Capacidad_MWh']
            pv_serie = pv_pu * cfg['PV_MW']
            c_deg = costo_degradacion_COP_MWh(pot, cap)
            for est in (ESTRATEGIAS if cap > 0 else ESTRATEGIAS[:1]):
                d = despachar(pv_serie, load, c_imp, c_exp, est,
                              power_MW=pot, cap_MWh=cap, c_deg=c_deg)
                fin = evaluar_finanzas(d, c_imp, c_exp, cfg['PV_MW'], pot, cap)
                if mejor is None or fin['VPN_COP'] > mejor['VPN_COP']:
                    mejor = {**cfg, 'Estrategia': est, **fin}
        salida[nombre_tarifa] = mejor
    return salida


def validar_robustez_inputs(pasos):
    """Repite la busqueda completa para cada combinacion de series de entrada.

    `pasos` es una lista de (etiqueta, pv_pu, load). Todas las series entran por
    argumento y se copian antes de usarse, de modo que la validacion no puede
    dejar tocados los vectores globales del barrido principal.
    """
    filas = []
    for etiqueta, pv_pu, load in pasos:
        print(f'  recalculando: {etiqueta} ...')
        for nombre_tarifa, m in mejor_por_tarifa(np.array(pv_pu, copy=True),
                                                 np.array(load, copy=True)).items():
            filas.append({
                'Paso': etiqueta, 'Tarifa': nombre_tarifa,
                'PV': m['PV_MW'], 'P_BESS': m['Potencia_MW'],
                'E_BESS': m['Capacidad_MWh'], 'Duracion': m['Duracion_h'],
                'Estrategia': m['Estrategia'], 'VPN': m['VPN_COP'] / 1e9,
                'TIR': m['TIR'],
            })
    return pd.DataFrame(filas)


if not RUN_VALIDACION_HISTORICA:
    EVOLUCION = None
    print('Validación histórica omitida.')
    print('Activar RUN_VALIDACION_HISTORICA=True para reproducir la prueba de '
          'robustez de inputs.')
else:
    import time as _time
    _t = _time.time()

    # Copias de referencia para comprobar que la validacion no deja efectos.
    _pv_ref, _load_ref = PV_PU.copy(), LOAD.copy()

    EVOLUCION = validar_robustez_inputs([
        ('A. Oriente.xlsx + demanda cruda', PV_PU_LEGADO, LOAD_CRUDO),
        ('B. TMY + demanda cruda', PV_PU, LOAD_CRUDO),
        ('C. TMY + demanda limpia', PV_PU, LOAD),
    ])

    # A3: los vectores del modelo principal siguen intactos.
    assert np.array_equal(PV_PU, _pv_ref), 'La validación alteró PV_PU'
    assert np.array_equal(LOAD, _load_ref), 'La validación alteró LOAD'
    del _pv_ref, _load_ref

    # El paso C es el mismo estado que el barrido principal: tienen que coincidir.
    # Es la comprobacion cruzada de que `mejor_por_tarifa` reproduce el barrido.
    for nombre_tarifa in TARIFAS:
        _c = EVOLUCION[(EVOLUCION.Paso.str.startswith('C.'))
                       & (EVOLUCION.Tarifa == nombre_tarifa)].iloc[0]
        _r = R[R.Tarifa == nombre_tarifa].nlargest(1, 'VPN_mil_MM').iloc[0]
        assert (_c.PV, _c.P_BESS, _c.E_BESS) == (_r.PV_MW, _r.Potencia_MW,
                                                 _r.Capacidad_MWh), (
            f'{nombre_tarifa}: el paso C no coincide con el barrido principal')
        assert abs(_c.VPN - _r.VPN_mil_MM) < 1e-9, (
            f'{nombre_tarifa}: el VPN del paso C no coincide con el barrido')

    print(f'  listo en {_time.time() - _t:.0f}s')
    print()
    print('=' * 92)
    print('EVOLUCION DEL DIMENSIONAMIENTO CON LAS CORRECCIONES DE DATOS')
    print('=' * 92)

    for nombre_tarifa in TARIFAS:
        t = EVOLUCION[EVOLUCION.Tarifa == nombre_tarifa].drop(columns='Tarifa')
        print(f'\n{nombre_tarifa}')
        display(t.style.format({
            'PV': '{:,.0f}', 'P_BESS': '{:,.0f}', 'E_BESS': '{:,.0f}',
            'Duracion': '{:,.1f}', 'VPN': '{:,.1f}', 'TIR': '{:.1%}'})
            .hide(axis='index'))

    _geoms = {(r.PV, r.P_BESS, r.E_BESS) for r in EVOLUCION.itertuples()}
    print()
    if len(_geoms) == 1:
        pv, p, e = next(iter(_geoms))
        print(f'La geometria PV {pv:.0f} MW + BESS {p:.0f} MW / {e:.0f} MWh gana en')
        print('los tres pasos y bajo las dos tarifas: es estable frente a las dos')
        print('correcciones independientes de los datos de entrada.')
    else:
        print(f'Aparecen {len(_geoms)} geometrias ganadoras distintas a lo largo de')
        print('los pasos: el dimensionamiento NO es estable frente a los inputs.')
        for g in sorted(_geoms):
            print(f'   PV {g[0]:.0f} MW + BESS {g[1]:.0f} MW / {g[2]:.0f} MWh')

Validación histórica omitida.
Activar RUN_VALIDACION_HISTORICA=True para reproducir la prueba de robustez de inputs.


## 10.5 ¿El óptimo toca un borde de la malla?

Un óptimo pegado al borde no es un óptimo: es el mejor de los que se probaron,
y no se sabe si más allá hay algo mejor. Es el defecto documentado en
`INFORME` §7.1, y aquí se convierte en una comprobación explícita.

Se revisan cuatro ejes —PV, potencia, energía y duración— para el mejor
resultado de cada tarifa y de cada estrategia. **La malla no se amplía
automáticamente**: por ahora solo se identifica si ocurre.

Si alguna casilla sale en `True`, ese resultado **no debe declararse todavía
como tamaño óptimo definitivo**.

In [28]:
# Extremos de la malla, leidos de la malla misma y no escritos a mano.
PV_MAX = max(PV_OPCIONES)
P_MAX = max(b['Potencia_MW'] for b in BESS_OPCIONES)
E_MAX = max(b['Capacidad_MWh'] for b in BESS_OPCIONES)
DUR_MIN = min(b['Duracion_h'] for b in BESS_OPCIONES if b['Potencia_MW'] > 0)
DUR_MAX = max(b['Duracion_h'] for b in BESS_OPCIONES if b['Potencia_MW'] > 0)


def marcar_bordes(fila):
    """Cuatro banderas de borde para una fila ganadora."""
    con_bess = fila.Capacidad_MWh > 0
    return {
        'Optimo_en_borde_PV': bool(fila.PV_MW == PV_MAX),
        'Optimo_en_borde_P': bool(con_bess and fila.Potencia_MW == P_MAX),
        'Optimo_en_borde_E': bool(con_bess and fila.Capacidad_MWh == E_MAX),
        'Optimo_en_borde_duracion': bool(
            con_bess and fila.Duracion_h in (DUR_MIN, DUR_MAX)),
    }


filas_borde = []
for nombre_tarifa in TARIFAS:
    sub = R[R.Tarifa == nombre_tarifa]

    mejor = sub.nlargest(1, 'VPN_mil_MM').iloc[0]
    filas_borde.append({'Tarifa': nombre_tarifa, 'Ambito': 'global',
                        'PV_MW': mejor.PV_MW, 'P_MW': mejor.Potencia_MW,
                        'E_MWh': mejor.Capacidad_MWh, 'Dur_h': mejor.Duracion_h,
                        **marcar_bordes(mejor)})

    for est in ESTRATEGIAS:
        m = sub[sub.Estrategia == est].nlargest(1, 'VPN_mil_MM').iloc[0]
        filas_borde.append({'Tarifa': nombre_tarifa, 'Ambito': est,
                            'PV_MW': m.PV_MW, 'P_MW': m.Potencia_MW,
                            'E_MWh': m.Capacidad_MWh, 'Dur_h': m.Duracion_h,
                            **marcar_bordes(m)})

BORDES = pd.DataFrame(filas_borde)

print('=' * 92)
print('OPTIMOS EN BORDE DE LA MALLA')
print('=' * 92)
print(f'Extremos: PV {PV_MAX:.0f} MW | P {P_MAX:.0f} MW | E {E_MAX:.0f} MWh | '
      f'duracion {DUR_MIN:.2f}-{DUR_MAX:.2f} h')
print()
display(BORDES.style.format({'PV_MW': '{:,.0f}', 'P_MW': '{:,.0f}',
                             'E_MWh': '{:,.0f}', 'Dur_h': '{:,.2f}'})
               .hide(axis='index'))

cols_borde = [c for c in BORDES.columns if c.startswith('Optimo_en_borde')]
tocados = BORDES[BORDES[cols_borde].any(axis=1)]
print()
if tocados.empty:
    print('Ningun optimo toca un borde: la malla contiene sus propios maximos.')
else:
    print(f'{len(tocados)} de {len(BORDES)} optimos tocan al menos un borde:')
    for _, f in tocados.iterrows():
        cuales = [c.replace('Optimo_en_borde_', '') for c in cols_borde if f[c]]
        print(f"   {f.Tarifa} / {f.Ambito}: borde en {', '.join(cuales)}")
    print()
    print('Esos resultados NO son tamanos optimos definitivos. Para cerrarlos hay')
    print('que ampliar el eje senalado y repetir el barrido.')

OPTIMOS EN BORDE DE LA MALLA
Extremos: PV 150 MW | P 60 MW | E 320 MWh | duracion 1.00-8.00 h



Tarifa,Ambito,PV_MW,P_MW,E_MWh,Dur_h,Optimo_en_borde_PV,Optimo_en_borde_P,Optimo_en_borde_E,Optimo_en_borde_duracion
T0 — señal cuasi-plana,global,150,40,320,8.00,True,False,True,True
T0 — señal cuasi-plana,A Autoconsumo,150,40,320,8.00,True,False,True,True
T0 — señal cuasi-plana,B Merito diario,150,40,320,8.00,True,False,True,True
T0 — señal cuasi-plana,C Merchant,100,20,80,4.00,False,False,False,False
T0 — señal cuasi-plana,D Merito+Export,150,40,320,8.00,True,False,True,True
T1 — señal horaria indexada,global,150,40,320,8.00,True,False,True,True
T1 — señal horaria indexada,A Autoconsumo,150,40,320,8.00,True,False,True,True
T1 — señal horaria indexada,B Merito diario,150,40,320,8.00,True,False,True,True
T1 — señal horaria indexada,C Merchant,100,10,80,8.00,False,False,False,True
T1 — señal horaria indexada,D Merito+Export,150,40,320,8.00,True,False,True,True



9 de 10 optimos tocan al menos un borde:
   T0 — señal cuasi-plana / global: borde en PV, E, duracion
   T0 — señal cuasi-plana / A Autoconsumo: borde en PV, E, duracion
   T0 — señal cuasi-plana / B Merito diario: borde en PV, E, duracion
   T0 — señal cuasi-plana / D Merito+Export: borde en PV, E, duracion
   T1 — señal horaria indexada / global: borde en PV, E, duracion
   T1 — señal horaria indexada / A Autoconsumo: borde en PV, E, duracion
   T1 — señal horaria indexada / B Merito diario: borde en PV, E, duracion
   T1 — señal horaria indexada / C Merchant: borde en duracion
   T1 — señal horaria indexada / D Merito+Export: borde en PV, E, duracion

Esos resultados NO son tamanos optimos definitivos. Para cerrarlos hay
que ampliar el eje senalado y repetir el barrido.


## 10.6 Mejor batería para cada tamaño de PV

`Delta_VPN_vs_solo_PV` es la columna que importa: separa el valor del
**almacenamiento** del valor de instalar **más paneles**. Sin ella es fácil
atribuir a la batería una mejora que en realidad viene del PV.

$$\Delta VPN_{BESS} = VPN(PV + BESS) - VPN(PV\ solo)$$

El caso `PV = 0` no tiene contraparte «solo PV» en la malla —no invertir nada
vale cero—, así que su línea base es 0.

In [29]:
filas_pv = []
for nombre_tarifa in TARIFAS:
    sub = R[R.Tarifa == nombre_tarifa]
    con_bess = sub[sub.Capacidad_MWh > 0]

    for pv in sorted(sub.PV_MW.unique()):
        cand = con_bess[con_bess.PV_MW == pv]
        if cand.empty:
            continue
        m = cand.nlargest(1, 'VPN_mil_MM').iloc[0]

        # Linea base: el mismo PV sin bateria. Con PV = 0 no existe esa fila
        # -no invertir nada vale cero-, asi que la base es 0.
        solo = sub[(sub.PV_MW == pv) & (sub.Capacidad_MWh == 0)].VPN_mil_MM
        base = float(solo.max()) if len(solo) else 0.0

        filas_pv.append({
            'Tarifa': nombre_tarifa, 'PV_MW': pv,
            'P_BESS_MW': m.Potencia_MW, 'E_BESS_MWh': m.Capacidad_MWh,
            'Duracion_h': m.Duracion_h, 'Estrategia': m.Estrategia,
            'VPN': m.VPN_mil_MM, 'VPN_solo_PV': base,
            'Delta_VPN_vs_solo_PV': m.VPN_mil_MM - base,
        })

POR_PV = pd.DataFrame(filas_pv)

for nombre_tarifa in TARIFAS:
    t = POR_PV[POR_PV.Tarifa == nombre_tarifa].drop(columns='Tarifa')
    print('=' * 92)
    print(f'MEJOR BESS PARA CADA TAMANO DE PV - {nombre_tarifa}')
    print('=' * 92)
    display(t.style.format({
        'PV_MW': '{:,.0f}', 'P_BESS_MW': '{:,.0f}', 'E_BESS_MWh': '{:,.0f}',
        'Duracion_h': '{:,.2f}', 'VPN': '{:,.1f}', 'VPN_solo_PV': '{:,.1f}',
        'Delta_VPN_vs_solo_PV': '{:+,.1f}'})
        .background_gradient(subset=['Delta_VPN_vs_solo_PV'], cmap='RdYlGn')
        .hide(axis='index'))

print('VPN en mil millones de COP. Delta positivo = la bateria suma sobre el')
print('mismo PV sin bateria; negativo = destruye valor a estos precios.')


MEJOR BESS PARA CADA TAMANO DE PV - T0 — señal cuasi-plana


PV_MW,P_BESS_MW,E_BESS_MWh,Duracion_h,Estrategia,VPN,VPN_solo_PV,Delta_VPN_vs_solo_PV
0,5,20,4.00,A Autoconsumo,-13.8,0.0,-13.8
10,5,20,4.00,B Merito diario,32.1,45.7,-13.5
20,5,20,4.00,A Autoconsumo,80.5,93.7,-13.2
30,5,20,4.00,A Autoconsumo,128.3,136.9,-8.6
40,5,20,4.00,A Autoconsumo,164.1,162.8,+1.4
50,10,40,4.00,A Autoconsumo,187.9,176.8,+11.1
70,20,80,4.00,A Autoconsumo,225.0,189.7,+35.3
100,40,240,6.00,A Autoconsumo,264.1,193.9,+70.2
150,40,320,8.00,B Merito diario,315.5,186.2,+129.3


MEJOR BESS PARA CADA TAMANO DE PV - T1 — señal horaria indexada


PV_MW,P_BESS_MW,E_BESS_MWh,Duracion_h,Estrategia,VPN,VPN_solo_PV,Delta_VPN_vs_solo_PV
0,5,20,4.00,C Merchant,-11.7,0.0,-11.7
10,5,20,4.00,C Merchant,32.3,43.8,-11.5
20,5,20,4.00,C Merchant,78.7,90.0,-11.3
30,5,20,4.00,B Merito diario,124.5,131.3,-6.8
40,10,20,2.00,B Merito diario,162.4,155.4,+6.9
50,20,80,4.00,B Merito diario,193.8,168.3,+25.5
70,20,80,4.00,B Merito diario,236.3,179.6,+56.7
100,40,240,6.00,B Merito diario,282.4,182.5,+99.9
150,40,320,8.00,B Merito diario,325.0,173.7,+151.4


VPN en mil millones de COP. Delta positivo = la bateria suma sobre el
mismo PV sin bateria; negativo = destruye valor a estos precios.


## 10.7 El óptimo propio de cada estrategia

Cada regla de operación tiene su propio $(PV^*, P^*, E^*)$. Si difieren entre
sí, el dimensionamiento y la estrategia **no son separables**: no se puede
dimensionar primero y escoger la regla después.

Es la base del estudio de interacción que viene más adelante.

In [30]:
filas_est = []
for nombre_tarifa in TARIFAS:
    sub = R[R.Tarifa == nombre_tarifa]
    for est in ESTRATEGIAS:
        m = sub[sub.Estrategia == est].nlargest(1, 'VPN_mil_MM').iloc[0]
        filas_est.append({
            'Tarifa': nombre_tarifa, 'Estrategia': est,
            'PV_optimo': m.PV_MW, 'P_BESS_optimo': m.Potencia_MW,
            'E_BESS_optimo': m.Capacidad_MWh, 'Duracion_optima': m.Duracion_h,
            'VPN_max': m.VPN_mil_MM, 'TIR': m.TIR, 'Ciclos_anio': m.Ciclos_anio,
        })

POR_ESTRATEGIA = pd.DataFrame(filas_est)

print('=' * 92)
print('OPTIMO PROPIO DE CADA ESTRATEGIA')
print('=' * 92)
display(POR_ESTRATEGIA.style.format({
    'PV_optimo': '{:,.0f}', 'P_BESS_optimo': '{:,.0f}',
    'E_BESS_optimo': '{:,.0f}', 'Duracion_optima': '{:,.2f}',
    'VPN_max': '{:,.1f}', 'TIR': '{:.1%}', 'Ciclos_anio': '{:,.0f}'})
    .hide(axis='index'))

for nombre_tarifa in TARIFAS:
    t = POR_ESTRATEGIA[POR_ESTRATEGIA.Tarifa == nombre_tarifa]
    geometrias = {(r.PV_optimo, r.P_BESS_optimo, r.E_BESS_optimo)
                  for r in t.itertuples()}
    print()
    if len(geometrias) == 1:
        pv, p, e = next(iter(geometrias))
        print(f'{nombre_tarifa}: las cuatro estrategias comparten el mismo optimo '
              f'(PV {pv:.0f} MW, BESS {p:.0f} MW / {e:.0f} MWh).')
        print('   El dimensionamiento se puede fijar antes que la regla.')
    else:
        print(f'{nombre_tarifa}: {len(geometrias)} geometrias optimas distintas '
              f'entre las {len(ESTRATEGIAS)} estrategias.')
        print('   Dimensionamiento y estrategia NO son separables.')

OPTIMO PROPIO DE CADA ESTRATEGIA


Tarifa,Estrategia,PV_optimo,P_BESS_optimo,E_BESS_optimo,Duracion_optima,VPN_max,TIR,Ciclos_anio
T0 — señal cuasi-plana,A Autoconsumo,150,40,320,8.00,315.5,19.0%,300
T0 — señal cuasi-plana,B Merito diario,150,40,320,8.00,315.5,19.0%,300
T0 — señal cuasi-plana,C Merchant,100,20,80,4.00,222.7,21.7%,357
T0 — señal cuasi-plana,D Merito+Export,150,40,320,8.00,310.4,18.8%,309
T1 — señal horaria indexada,A Autoconsumo,150,40,320,8.00,324.1,19.2%,300
T1 — señal horaria indexada,B Merito diario,150,40,320,8.00,325.0,19.2%,300
T1 — señal horaria indexada,C Merchant,100,10,80,8.00,193.0,20.4%,320
T1 — señal horaria indexada,D Merito+Export,150,40,320,8.00,319.8,19.1%,309



T0 — señal cuasi-plana: 2 geometrias optimas distintas entre las 4 estrategias.
   Dimensionamiento y estrategia NO son separables.

T1 — señal horaria indexada: 2 geometrias optimas distintas entre las 4 estrategias.
   Dimensionamiento y estrategia NO son separables.


### Alcance de esta corrida

Esta ejecución cambia **únicamente el espacio de dimensionamiento**. Sigue
usando el modelo financiero actual, así que la lectura debe limitarse a:

- cómo se mueve el óptimo al liberar MW y MWh;
- si 20 MW / 80 MWh sigue siendo competitivo;
- qué duraciones aparecen;
- si alguna solución cae en un borde;
- cuánto cambia el tamaño óptimo entre estrategias.

**No son conclusiones definitivas de rentabilidad.** Quedan pendientes de
revisión, y cualquiera puede mover el resultado:

1. degradación del PV aplicada al flujo financiero;
2. costo sombra de degradación del BESS;
3. vida y reposición del BESS;
4. límites del punto de conexión;
5. consistencia de las curvas de CAPEX.

## 11. Conclusiones

La celda siguiente arma el reporte final. Los puntos que conviene mirar:

1. **Cuál es la configuración óptima** y cuánto VPN produce frente al CAPEX que exige.
2. **Cuánto pesa el dimensionamiento frente a la estrategia.** Si la razón es de
   uno o dos órdenes de magnitud, el mensaje práctico es que la decisión de
   inversión manda sobre la de operación.
3. **Si la estrategia óptima cambia con la tarifa.** Es el argumento de que
   estrategia y contexto regulatorio no se pueden separar.
4. **Qué tan robusta es la conclusión.** Los puntos de quiebre dicen cuánto margen
   hay antes de que el proyecto deje de pagarse.

### Limitaciones que conviene declarar en el documento de tesis

- **Un solo año de datos (2025).** No captura variabilidad hidrológica entre años,
  que en Colombia mueve fuertemente el precio de bolsa. Repetir con varios años,
  o al menos con un año seco y uno húmedo, cambiaría los resultados.
- **Previsión perfecta dentro del día** en las estrategias B y D. Un pronóstico
  real tiene error, así que su desempeño es una cota superior.
- **CAPEX y vida útil son valores de referencia**, no cotizaciones. Los puntos de
  quiebre de la sección 10 indican cuánto importa afinarlos.
- **No se modelan servicios complementarios** (AGC, reserva), que son fuentes
  de ingreso reales para un BESS y que aquí valen cero. El cargo por potencia,
  en cambio, no es una omisión: la tarifa colombiana para usuarios finales del
  SIN es monomia y no tiene término de $/kW-mes.
- **No se cobra la capacidad del punto de conexión.** El modelo la dimensiona
  como `demanda máxima + potencia del BESS` sin costo asociado; es una decisión
  de alcance declarada, no un descuido.
- **El beneficio operativo se escala por degradación** en vez de re-simular cada
  año, tal como se explicó en la sección 6.

In [31]:
linea = '=' * 92
salida = [linea, 'REPORTE FINAL — OPTIMIZACIÓN DE PORTAFOLIO PV + BESS', linea, '']

salida.append(f'Espacio explorado : {len(configuraciones)} configuraciones '
              f'x {len(ESTRATEGIAS)} estrategias x {len(TARIFAS)} tarifas')
salida.append(f'Horizonte         : {HORIZONTE_ANIOS} años | WACC {WACC:.0%}')
salida.append(f'Validaciones      : {len(R):,} evaluaciones, {len(fallos)} fallos físicos')
salida.append('')

for nombre_tarifa in TARIFAS:
    sub = R[R.Tarifa == nombre_tarifa]
    mejor = sub.nlargest(1, 'VPN_mil_MM').iloc[0]
    solo_pv = sub[(sub.PV_MW == mejor.PV_MW) & (sub.Capacidad_MWh == 0)].VPN_mil_MM.max()

    salida += [linea, f'{nombre_tarifa}', linea]
    salida.append(f'  Mejor config. hallada: PV {mejor.PV_MW:,.0f} MW  +  BESS '
                  f'{mejor.Potencia_MW:,.0f} MW / {mejor.Capacidad_MWh:,.0f} MWh')
    salida.append(f'  Estrategia óptima    : {mejor.Estrategia}')
    salida.append(f'  VPN                  : {mejor.VPN_mil_MM:>10,.1f} mil MM COP')
    salida.append(f'  TIR                  : {mejor.TIR:>10.1%}')
    salida.append(f'  CAPEX                : {mejor.CAPEX_mil_MM:>10,.1f} mil MM COP')
    salida.append(f'  Payback descontado   : {mejor.Payback_anios:>10,.0f} años')
    salida.append('  Costo nivelado       : NO REPORTAR - pendiente de modelo multianual')
    if mejor.Capacidad_MWh > 0:
        salida.append(f'  Aporte del BESS      : {mejor.VPN_mil_MM - solo_pv:>10,.1f} mil MM COP '
                      f'(VPN solo con PV: {solo_pv:,.1f})')
    else:
        mejor_con_bess = sub[sub.Capacidad_MWh > 0].nlargest(1, 'VPN_mil_MM').iloc[0]
        salida.append(f'  BESS                 : NO entra en el portafolio óptimo')
        salida.append(f'     mejor caso con BESS: {mejor_con_bess.Potencia_MW:,.0f} MW / '
                      f'{mejor_con_bess.Capacidad_MWh:,.0f} MWh -> VPN '
                      f'{mejor_con_bess.VPN_mil_MM:,.1f} mil MM COP '
                      f'({mejor_con_bess.VPN_mil_MM - mejor.VPN_mil_MM:+,.1f} vs. solo PV)')

    con_bess = sub[sub.Capacidad_MWh > 0]
    rango_est = (con_bess.groupby(['PV_MW', 'Potencia_MW', 'Capacidad_MWh'])
                          .VPN_mil_MM.agg(lambda s: s.max() - s.min()).median())
    rango_dim = con_bess.VPN_mil_MM.max() - con_bess.VPN_mil_MM.min()
    salida.append(f'  Dimensionamiento manda sobre estrategia por '
                  f'{rango_dim/max(rango_est,1e-9):,.0f}x '
                  f'({rango_dim:,.0f} vs {rango_est:,.1f} mil MM COP)')
    salida.append('')

mejores = {t: R[R.Tarifa == t].nlargest(1, 'VPN_mil_MM').iloc[0] for t in TARIFAS}
if len({m.Estrategia for m in mejores.values()}) > 1:
    salida.append('>> La estrategia óptima CAMBIA con la estructura tarifaria:')
    for t, m in mejores.items():
        salida.append(f'     {t}: {m.Estrategia}')
    salida.append('   La regla de operación no se puede escoger sin fijar el contrato.')
else:
    unica = next(iter(mejores.values())).Estrategia
    salida.append(f'>> La misma estrategia ({unica}) gana bajo ambas tarifas.')
    salida.append('   El resultado es robusto a la estructura tarifaria.')

if len({(m.PV_MW, m.Capacidad_MWh) for m in mejores.values()}) > 1:
    salida.append('')
    salida.append('>> El dimensionamiento óptimo también cambia con la tarifa:')
    for t, m in mejores.items():
        salida.append(f'     {t}: PV {m.PV_MW:,.0f} MW + BESS {m.Capacidad_MWh:,.0f} MWh')

salida += ['', 'Nota: "mejor" = maximo de VPN sobre las configuraciones evaluadas,',
           'operadas con reglas heuristicas. El despacho matematicamente optimo',
           'no se calcula aqui; esa cota vive en el cuaderno 01 con PyPSA.', linea]
print('\n'.join(salida))

REPORTE FINAL — OPTIMIZACIÓN DE PORTAFOLIO PV + BESS

Espacio explorado : 233 configuraciones x 4 estrategias x 2 tarifas
Horizonte         : 20 años | WACC 10%
Validaciones      : 1,864 evaluaciones, 0 fallos físicos

T0 — señal cuasi-plana
  Mejor config. hallada: PV 150 MW  +  BESS 40 MW / 320 MWh
  Estrategia óptima    : B Merito diario
  VPN                  :      315.5 mil MM COP
  TIR                  :      19.0%
  CAPEX                :      479.4 mil MM COP
  Payback descontado   :          8 años
  Costo nivelado       : NO REPORTAR - pendiente de modelo multianual
  Aporte del BESS      :      129.3 mil MM COP (VPN solo con PV: 186.2)
  Dimensionamiento manda sobre estrategia por 79x (537 vs 6.8 mil MM COP)

T1 — señal horaria indexada
  Mejor config. hallada: PV 150 MW  +  BESS 40 MW / 320 MWh
  Estrategia óptima    : B Merito diario
  VPN                  :      325.0 mil MM COP
  TIR                  :      19.2%
  CAPEX                :      479.4 mil MM COP
  Payback 